# Class Balancing and Calibration for Toxicity Prediction on New Chemical Scaffolds

**Muhammad Zohaib Chaudhary**
**CSC606 Machine Learning — Final Project (Phase 2)**

**Project repository:** <https://github.com/mzohaibcc/project606>

---

### Abstract

Toxicity screening datasets are severely imbalanced: in the three Tox21 endpoints studied
here, between 4.3% and 15.9% of tested compounds are active. The standard response is to
rebalance the training data, but a screening programme needs two things at once — it must
rank the right compounds into a limited experimental budget, *and* its predicted
probabilities must mean what they say. This project asks whether class balancing delivers
the first without destroying the second, and whether any benefit survives when the test
compounds come from chemical scaffolds the model has never seen. A fixed grid of 144 models
— 3 assays x 2 classifiers (regularized logistic regression, random forest) x 4 balancing
choices (none, inverse-frequency class weights, random undersampling, SMOTEENN) x 2 split
designs (Bemis-Murcko scaffold hold-out, stratified random) x 3 seeds — is trained on
2,048-bit Morgan fingerprints, tuned by cross-validated average precision inside the
training partition only, and scored on a held-out test partition at a fixed 10% screening
budget, before and after sigmoid calibration fitted on a separate calibration partition.
The results reproduced in this notebook show that resampling methods reduced detection more
often than they improved it; that inverse-frequency class weighting was the only balancing
choice that did not systematically cost detection, with its one interval-supported gain
confined to the random forest on the scaffold split and absent on one of the three assays;
that balancing badly degrades probability quality, with 84 of 144 raw models scoring worse
than a constant-prevalence predictor, which post-hoc calibration then repairs for all 144;
and that calibration cannot and did not change any ranking metric. Testing on held-out
scaffolds was harder in all 24 assay x classifier x method cells, and across-seed
variability more than doubled.

> **Scope.** An active assay result is an experimental measurement, not a determination that
> a chemical is harmful to people. Everything below concerns model ranking performance,
> probability calibration, descriptive uncertainty, and the limits of this experimental
> design.

---

### How to run this notebook

**On Google Colab — the intended path.** Open the shared link and choose
**Runtime > Run all**. No file upload is required and nothing needs to be configured.
Section 2.1 clones the public repository above into the session, and section 2.2 installs
the exact dependency versions the committed results were produced with. If Colab reports
that a session restart is needed after the install, choose **Runtime > Restart session**
and then **Run all** again — every cell is written to be idempotent, so a second pass is
safe. On a typical Colab CPU runtime expect roughly 8 to 15 minutes end to end, most of it
the clone and the dependency install; the exact time depends on the runtime you are given.

**Locally.** Launch the notebook from inside a checkout of the repository using the project
virtual environment. The Colab-only cells detect that `google.colab` is absent and skip
themselves; section 2.1 then locates the project by walking up from the working directory.

**What this notebook writes.** Nothing in `data/`, `partitions/`, `outputs/full/` or
`outputs/smoke/` is ever written to. The one section that executes the pipeline directs its
output to the scratch directory `outputs/notebook_demo/`, asserts that before it starts, and
the final cell verifies afterwards that no protected file was touched.

## 1. The problem, the design, and what this notebook contains

### 1.1 Research question

*Which balancing methods help models find more active compounds on new chemical scaffolds
while keeping their probability estimates reliable?*

Every competing method is given the same screening budget — the top 10% of ranked test
compounds — and the predicted probabilities of each method are examined before and after
sigmoid calibration fitted on a separate calibration partition. Two sub-questions follow
from the main one: does any balancing benefit survive the move from a random split to a
scaffold hold-out, and does post-hoc calibration repair the probability damage that
balancing causes?

### 1.2 Why it matters

A high-throughput screen can only follow up a fixed fraction of a compound library, so the
useful question is not overall accuracy — a model that predicts "inactive" for everything
scores between 84.1% and 95.7% accuracy on these assays and finds nothing. The useful questions are how
many actives land inside the budget, and whether a predicted probability of 0.3 really
corresponds to a 30% chance of activity, because that is what a triage decision rests on.
Class imbalance is known to distort both, and the usual remedies are themselves known to
distort predicted probabilities. This project measures both effects on the same models.

### 1.3 Experimental design

The benchmark is a fixed grid, specified before any result was seen:

| Axis | Levels |
| --- | --- |
| Assays | NR-AR, NR-AhR, SR-MMP (Tox21) |
| Classifiers | regularized logistic regression, random forest |
| Balancing | none, inverse-frequency class weights, random undersampling, SMOTEENN |
| Split design | Bemis-Murcko scaffold hold-out, stratified random reference |
| Seeds | 17, 42, 2026 |

3 x 2 x 4 x 2 x 3 = **144 fitted models**. The exact grid is printed from
`configs/full.json` in section 2.5 rather than restated here from memory.

Each molecule is represented as a 2,048-bit Morgan fingerprint of radius 2 (ECFP4-style).
Hyperparameters are chosen by mean average precision over three cross-validation folds
**inside the training partition only**; resampling and class weights are fitted inside each
fold, never on the whole training set before splitting. The calibration partition and the
test partition never inform model or sampler selection, which is how this design avoids
leakage from the final test data into training or model choice.

### 1.4 What this notebook contains

| Section | Contents |
| --- | --- |
| 1 | Problem, research question, experimental design |
| 2 | Environment: clone the repository, install the pinned dependencies, verify versions, print the design grid |
| 3 | Data loading and preprocessing: provenance, cleaning audit, and the standardization and fingerprint code run live |
| 4 | Partition design: realised fractions, positives, the scaffold caveat, and the splitter re-run live |
| 5 | The training implementation printed from the project source, then **live re-derivation**: the pipeline is actually executed and compared against the committed run |
| 6 | Detection results: seed means and SDs, paired bootstrap deltas, and the tally across all comparisons |
| 7 | Calibration results: ranking invariance, Brier and log loss before and after, reliability gaps |
| 8 | Scaffold versus random: does any benefit survive the harder split? |
| 9 | Figures: panels regenerated here, plus the committed precision-recall and reliability panels |
| 10 | Similarity analysis over fixed nearest-neighbour Tanimoto bins |
| 11 | Documented limitations |
| 12 | Headline summary and a write-isolation check |

### 1.5 How to read the numbers

Tables keep a fixed method order (`none, weights, undersample, smoteenn`) and a fixed
calibration order (`raw, sigmoid`). No table is sorted by a metric and no row is highlighted
as a winner. Higher is better for average precision, ROC-AUC, and precision and recall at
the 10% budget; **lower** is better for Brier score and log loss. Bootstrap intervals are
descriptive and conditional — section 11 sets out exactly what they do and do not support.

### 1.6 Note on the cited benchmark

The evaluation layout follows the setup described by the ToxBench study (reference [4] in
the proposal: Kartic, Seo, Yi and Park, 2026). `docs/references.md` records that only the
bibliographic metadata and the publisher abstract of that article were verified, and that
**the proposal's specific assertion about its tree-model class weighting was not
independently verified and is not treated as confirmed anywhere in this notebook.**

## 2. Environment setup

Four cells follow. They (2.1) obtain the project tree, (2.2) install the pinned
dependencies on Colab, (2.3) check the installed versions against the pins and stop if they
disagree, and (2.4) import the project package and print the full version report. Section
2.5 then prints the experimental design from the committed configuration file.

### How this notebook handles dependencies, and why

The committed results were produced with an exact set of package versions, recorded in
`requirements.txt` and in `outputs/full/environment.json`. Google Colab ships its own
numpy, pandas, scikit-learn and matplotlib, and those versions drift over time. Two
approaches were possible: install only what Colab lacks, or install the full pinned set.

**This notebook installs the full pinned set**, for one reason: section 5 re-runs the
training pipeline live and compares its predictions against the committed ones. That
comparison is only meaningful if the numerical stack matches, because scikit-learn's solver
behaviour, numpy's reductions and RDKit's fingerprint generation all change between
versions. Installing only the missing packages would leave the live slice silently running
on a different stack and quietly producing different numbers — the one failure mode worth
spending install time to avoid.

The cost of that choice is that replacing numpy or pandas underneath a running Python kernel
does not take effect until the kernel restarts. Colab usually prompts for this. Cell 2.3
therefore compares three things for every pinned package: the version required, the version
now installed on disk, and the version actually loaded into this session. It distinguishes
the two failure modes and stops with a specific instruction for each:

* **a stale import** — the right version is on disk but an older one is already loaded. The
  fix is **Runtime > Restart session**, then **Run all**. This is the normal Colab case.
* **a pin that could not be installed** — pip could not supply the required version. The
  message names the package. Sections 6 to 11 read committed result files and are unaffected
  by package versions, so that cell also documents how to proceed past it deliberately.

Cell 2.3 uses only the standard library, and runs **before** numpy, pandas or RDKit are
imported, so a mismatched binary interface produces the readable message above rather than
an import crash.

In [ ]:
# --- 2.1 Obtain the project tree and make it the working directory --------------------
# On Colab this clones the public repository; locally it walks up from the working
# directory to find an existing checkout. Both paths end with PROJECT_ROOT set, that
# directory as the working directory, and that directory on sys.path.
#
# Only the standard library is used here: nothing from the project, numpy or pandas is
# imported until the dependency versions have been checked in cell 2.3.
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/mzohaibcc/project606.git"
REPO_DIRNAME = "project606"

IN_COLAB = "google.colab" in sys.modules
PROJECT_ROOT = None


def _looks_like_project(path):
    """A directory is the project root if it carries the runner and the full config."""
    path = Path(path)
    return (path / "toxbench" / "runner.py").is_file() and (path / "configs" / "full.json").is_file()


def activate(path):
    """Make `path` the project root: chdir into it and put it on sys.path."""
    global PROJECT_ROOT
    PROJECT_ROOT = Path(path).resolve()
    os.chdir(PROJECT_ROOT)
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    return PROJECT_ROOT


def find_existing_project():
    """Look for a checkout at or above the working directory, then in the clone target."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents, here / REPO_DIRNAME]:
        try:
            if _looks_like_project(candidate):
                return candidate
        except OSError:
            continue
    return None


def clone_project():
    """Clone the public repository beside the working directory. Idempotent."""
    target = (Path.cwd() / REPO_DIRNAME).resolve()
    if target.exists():
        if _looks_like_project(target):
            print("Repository already cloned at", target, "- reusing it.")
            return target
        # A partial clone from an interrupted run: only ever remove our own target name.
        print("Removing an incomplete clone at", target)
        shutil.rmtree(target)
    print("Cloning", REPO_URL)
    print("This downloads the code, the data, and the committed benchmark results.")
    completed = subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(target)],
        text=True, capture_output=True,
    )
    if completed.returncode != 0:
        print(completed.stdout.strip())
        print(completed.stderr.strip()[-2000:])
        raise RuntimeError(
            "git clone failed (exit {}). Check the Colab session has network access, then "
            "re-run this cell.".format(completed.returncode))
    print(completed.stderr.strip()[-500:] or completed.stdout.strip()[-500:])
    if not _looks_like_project(target):
        raise RuntimeError("The clone completed but {} does not contain toxbench/runner.py.".format(target))
    return target


_found = find_existing_project()
if _found is not None:
    activate(_found)
    print("Found an existing project checkout.")
elif IN_COLAB:
    activate(clone_project())
else:
    raise FileNotFoundError(
        "Could not locate the project root (a directory containing toxbench/runner.py and "
        "configs/full.json) at or above " + str(Path.cwd()) + ". Start this notebook from "
        "inside a checkout of " + REPO_URL + ", or run it on Google Colab, where it clones "
        "the repository automatically.")

print()
print("Running on Google Colab :", IN_COLAB)
print("Project root            :", PROJECT_ROOT)
print("Working directory       :", Path.cwd())
print("Project on sys.path     :", str(PROJECT_ROOT) in sys.path)
print()
print("Top-level contents of the project tree:")
for entry in sorted(PROJECT_ROOT.iterdir()):
    if not entry.name.startswith("."):
        print("   ", entry.name + ("/" if entry.is_dir() else ""))

In [ ]:
# --- 2.2 Install the pinned dependency versions (Colab only) --------------------------
# requirements.txt also pins pytest, nbformat and psutil, which are test and packaging
# tools; `toxbench` imports none of them, so reinstalling them under a live kernel would
# add risk for no benefit. Only the packages that affect the computed numbers are
# installed here. Every one of them is reported in cell 2.3.
REQUIREMENTS = PROJECT_ROOT / "requirements.txt"

# Distribution name -> module name, for packages whose import name differs.
RUNTIME_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "scikit-learn": "sklearn",
    "imbalanced-learn": "imblearn",
    "rdkit": "rdkit",
    "matplotlib": "matplotlib",
    "joblib": "joblib",
    "threadpoolctl": "threadpoolctl",
}

# Parse requirements.txt rather than restating the versions here.
PINS = {}
for line in REQUIREMENTS.read_text().splitlines():
    line = line.strip()
    if line and not line.startswith("#"):
        name, _, version = line.partition("==")
        PINS[name.strip()] = version.strip()

RUNTIME_PINS = {name: PINS[name] for name in RUNTIME_PACKAGES if name in PINS}

# numpy 2.5.5 and scipy 1.18.1 publish wheels for CPython 3.12 and newer; RDKit 2025.09.3
# publishes them up to CPython 3.14. Outside that window pip would fall back to building
# from source or resolving a different version, so say so plainly instead.
PYTHON_MIN, PYTHON_MAX_EXCLUSIVE = (3, 12), (3, 15)
print("Python in this session:", sys.version.split()[0])
if not (PYTHON_MIN <= sys.version_info[:2] < PYTHON_MAX_EXCLUSIVE):
    raise RuntimeError(
        "This project's pinned dependencies require Python >= {}.{} and < {}.{}; this "
        "session runs {}.{}. numpy {} and scipy {} publish wheels only for Python >= 3.12, "
        "and rdkit {} only up to Python 3.14.".format(
            *PYTHON_MIN, *PYTHON_MAX_EXCLUSIVE, *sys.version_info[:2],
            PINS.get("numpy"), PINS.get("scipy"), PINS.get("rdkit")))
print("Within the supported window Python >= {}.{}, < {}.{}.".format(*PYTHON_MIN, *PYTHON_MAX_EXCLUSIVE))
print()

if IN_COLAB:
    specs = ["{}=={}".format(name, version) for name, version in RUNTIME_PINS.items()]
    print("Installing the pinned runtime dependencies:")
    for spec in specs:
        print("   ", spec)
    print("\nOn a fresh Colab runtime this typically takes two to five minutes.")
    completed = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *specs],
        text=True, capture_output=True,
    )
    if completed.stdout.strip():
        print(completed.stdout.strip()[-1500:])
    if completed.returncode != 0:
        # A bare CalledProcessError hides the reason, so surface pip's own message.
        print("\n--- pip stderr (last 3000 characters) ---")
        print(completed.stderr.strip()[-3000:])
        raise RuntimeError(
            "pip install failed with exit code {} under Python {}. The message above is "
            "pip's own. Cell 2.3 will not be able to verify the pinned versions until this "
            "succeeds.".format(completed.returncode, sys.version.split()[0]))
    if completed.stderr.strip():
        print("\n--- pip warnings (last 1500 characters) ---")
        print(completed.stderr.strip()[-1500:])
    # Newly installed distributions are invisible to importlib.metadata without this.
    import importlib
    importlib.invalidate_caches()
    print("\nInstall finished. Cell 2.3 now checks what actually landed.")
else:
    print("Not running on Google Colab: skipping the pip install.")
    print("Local execution uses the project virtual environment, pinned by")
    print(PROJECT_ROOT / "requirements-lock.txt")

In [ ]:
# --- 2.3 Version gate: required, installed on disk, and loaded in this session --------
# This runs BEFORE numpy, pandas or RDKit are imported, so a binary-interface mismatch
# produces the readable message below rather than an import crash. Standard library only.
#
# Set this to True only if you have read the message this cell prints and have decided to
# continue with versions that differ from the pins. Sections 6 to 11 read committed result
# files and do not depend on package versions; the live re-derivation in section 5 does.
ALLOW_VERSION_DRIFT = False

import importlib.metadata as importlib_metadata

try:  # `packaging` ships with pip, so it is present on Colab and in the project venv.
    from packaging.version import InvalidVersion, Version

    def _same_version(left, right):
        """Compare versions numerically: rdkit reports 2025.09.3 for the pin 2025.9.3."""
        try:
            return Version(str(left)) == Version(str(right))
        except InvalidVersion:
            return str(left) == str(right)
except ModuleNotFoundError:  # Fallback with no third-party dependency at all.
    import re

    def _key(text):
        return tuple(int(p) if p.isdigit() else p
                     for p in re.split(r"[.\-+_]", str(text).strip()) if p)

    def _same_version(left, right):
        return _key(left) == _key(right)


rows, stale_imports, wrong_pin, missing = [], [], [], []
for dist_name, module_name in RUNTIME_PACKAGES.items():
    pinned = RUNTIME_PINS.get(dist_name)
    try:
        installed = importlib_metadata.version(dist_name)
    except importlib_metadata.PackageNotFoundError:
        installed = None
    # Only report a loaded version for a module already imported; importing it here would
    # be the very mistake this cell exists to catch.
    module = sys.modules.get(module_name)
    in_session = getattr(module, "__version__", "loaded, version unknown") if module else "not imported yet"

    if installed is None:
        missing.append(dist_name)
        verdict = "NOT INSTALLED"
    elif pinned is not None and not _same_version(installed, pinned):
        wrong_pin.append((dist_name, pinned, installed))
        verdict = "WRONG VERSION"
    elif module is not None and not _same_version(in_session, installed):
        stale_imports.append((dist_name, installed, in_session))
        verdict = "STALE IMPORT"
    else:
        verdict = "ok"
    rows.append((dist_name, pinned or "-", installed or "-", str(in_session), verdict))

header = ("package", "required", "installed on disk", "loaded in session", "status")
widths = [max(len(str(r[i])) for r in (list(rows) + [header])) for i in range(5)]
line = "  ".join("{:<%d}" % w for w in widths)
print("Pinned runtime dependencies")
print(line.format(*header))
print(line.format(*("-" * w for w in widths)))
for row in rows:
    print(line.format(*row))
print()

if missing:
    raise RuntimeError(
        "These pinned packages are not installed: {}. On Colab, re-run cell 2.2. Locally, "
        "run `pip install -r requirements.txt` in the project virtual environment."
        .format(", ".join(missing)))

if stale_imports:
    detail = "; ".join("{} has {} on disk but {} loaded".format(n, d, s) for n, d, s in stale_imports)
    raise RuntimeError(
        "A newer version was installed than the one this session has already loaded ({}).\n"
        "This is the normal Colab case after cell 2.2 replaced a preinstalled package.\n"
        "FIX: choose Runtime > Restart session, then Runtime > Run all. Every cell is "
        "idempotent, so the second pass re-uses the clone and the install and reaches this "
        "point in seconds.".format(detail))

if wrong_pin and not ALLOW_VERSION_DRIFT:
    detail = "; ".join("{} requires {} but {} is installed".format(n, p, i) for n, p, i in wrong_pin)
    raise RuntimeError(
        "Installed versions differ from the pins the committed results were produced with "
        "({}).\n"
        "The live re-derivation in section 5 would then be running on a different numerical "
        "stack, and its comparison against the committed predictions would not be "
        "meaningful. Sections 6 to 11 read committed result files and are unaffected.\n"
        "FIX: re-run cell 2.2 and check pip's output for a resolution conflict.\n"
        "To continue anyway, deliberately: set ALLOW_VERSION_DRIFT = True at the top of "
        "this cell and re-run it. Section 5 will then print a warning alongside its "
        "results.".format(detail))

VERSIONS_MATCH_PINS = not wrong_pin
if wrong_pin:
    print("WARNING: continuing with ALLOW_VERSION_DRIFT = True. Live results in section 5")
    print("         may differ numerically from the committed benchmark.")
else:
    print("All {} pinned runtime packages match, and nothing stale is loaded.".format(len(rows)))

In [ ]:
# --- 2.4 Imports, display helpers, and the full version report ------------------------
import hashlib
import json
import time

import numpy as np
import pandas as pd

import toxbench
from toxbench.evaluation import METHODS, CALIBRATIONS, METRIC_NAMES, create_figures, metrics

try:  # IPython is present on Colab and in Jupyter; it may be absent in a bare interpreter.
    from IPython.display import Image as _IPythonImage, display as _ipython_display
    HAS_IPYTHON = True
except ModuleNotFoundError:
    HAS_IPYTHON = False

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_colwidth", 60)

# Any figure a display cell cannot find is recorded here and asserted in section 12.
MISSING_FIGURES = []


def show(frame, title=None):
    """Render a DataFrame inline on Colab/Jupyter, or as plain text elsewhere."""
    if title:
        print(title)
    data = frame
    if isinstance(data, pd.DataFrame):
        if data.columns.name is not None:
            data = data.rename_axis(columns=None)
        if not isinstance(data.index, pd.MultiIndex) and data.index.name is None:
            data = data.reset_index(drop=True)
    if HAS_IPYTHON:
        _ipython_display(data)
    else:
        print(data.to_string())
    print()


def show_image(path, caption=None):
    """Render a PNG inline on Colab/Jupyter, or report its path elsewhere."""
    path = Path(path)
    if caption:
        print(caption)
    if not path.is_file():
        MISSING_FIGURES.append(str(path))
        print("  MISSING:", path)
        return
    if HAS_IPYTHON:
        # The committed panels are 300 dpi, so cap the on-screen width.
        _ipython_display(_IPythonImage(filename=str(path), width=900))
    else:
        print("  [not rendered outside IPython] {} ({:,} bytes)".format(path, path.stat().st_size))


# The report covers every pin, including the test and packaging tools cell 2.2 leaves alone.
report_rows = []
for name, pinned in PINS.items():
    try:
        installed = importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        installed = "not installed"
    report_rows.append({
        "package": name,
        "pinned": pinned,
        "installed": installed,
        "matches_pin": installed != "not installed" and _same_version(installed, pinned),
        "installed_by_cell_2.2": name in RUNTIME_PINS,
    })
versions = pd.DataFrame(report_rows)

print("Python           :", sys.version.split()[0])
print("Platform         :", sys.platform)
print("toxbench package :", Path(toxbench.__file__).parent)
print("Running on Colab :", IN_COLAB)
print()
show(versions, "Every version pinned in requirements.txt, against what is installed")
print("Packages matching their pin: {} of {}".format(int(versions.matches_pin.sum()), len(versions)))
print()

# The committed results record the stack they were produced with; compare against it.
recorded = json.loads((PROJECT_ROOT / "outputs" / "full" / "environment.json").read_text())
print("The committed benchmark in outputs/full/ was produced with:")
print("  python   :", recorded["python"])
print("  platform :", recorded["platform"])
for package, version in recorded["versions"].items():
    try:
        here = importlib_metadata.version(package)
    except importlib_metadata.PackageNotFoundError:
        here = "not installed"
    flag = "same" if here != "not installed" and _same_version(here, version) else "DIFFERS"
    print("  {:<17}: {:<12} (this session: {:<12} {})".format(package, version, here, flag))

### 2.5 The experimental design, read from the committed configuration

Rather than restating the design from memory, the cell below prints it from
`configs/full.json` — the configuration file that produced every committed result in
sections 6 to 10. `configs/smoke.json` is the single-partition subset that section 5
executes live.

In [ ]:
# --- 2.5 The design grid, read from the committed configuration files -----------------
FULL_CONFIG = json.loads((PROJECT_ROOT / "configs" / "full.json").read_text())
SMOKE_CONFIG = json.loads((PROJECT_ROOT / "configs" / "smoke.json").read_text())

ASSAYS = list(FULL_CONFIG["assays"])
SPLIT_KINDS = list(FULL_CONFIG["split_kinds"])
SEEDS = list(FULL_CONFIG["seeds"])
CLASSIFIERS = list(FULL_CONFIG["classifiers"])
BALANCING = list(FULL_CONFIG["balancing"])

design = pd.DataFrame([
    {"design axis": "assays", "count": len(ASSAYS), "values": ", ".join(ASSAYS)},
    {"design axis": "classifiers", "count": len(CLASSIFIERS), "values": ", ".join(CLASSIFIERS)},
    {"design axis": "balancing choices", "count": len(BALANCING), "values": ", ".join(BALANCING)},
    {"design axis": "split kinds", "count": len(SPLIT_KINDS), "values": ", ".join(SPLIT_KINDS)},
    {"design axis": "seeds", "count": len(SEEDS), "values": ", ".join(str(s) for s in SEEDS)},
])
show(design, "Design grid from configs/full.json")

print("Fitted model configurations = {} x {} x {} x {} x {} = {}".format(
    len(ASSAYS), len(CLASSIFIERS), len(BALANCING), len(SPLIT_KINDS), len(SEEDS),
    len(ASSAYS) * len(CLASSIFIERS) * len(BALANCING) * len(SPLIT_KINDS) * len(SEEDS)))
print()
print("Hyperparameter grids searched by cross-validation inside the training partition:")
print("  Logistic regression C      :", FULL_CONFIG["logistic_C"])
print("  Forest max_depth           :", FULL_CONFIG["forest_depth"])
print("  Forest min_samples_leaf    :", FULL_CONFIG["forest_min_leaf"])
print("  Forest trees               :", FULL_CONFIG["forest_trees"])
print("  Cross-validation folds     :", FULL_CONFIG["cv_folds"])
print()
print("Bootstrap replicates per comparison:", FULL_CONFIG["bootstrap_replicates"])
print("Protocol version                   :", FULL_CONFIG["protocol_version"])
print()
print("Balancing method order used in every table below:", list(METHODS))
print("Calibration states                              :", list(CALIBRATIONS))
print("Metrics recorded per model                      :", list(METRIC_NAMES))

## 3. Data loading and preprocessing

The input is the public compressed Tox21 CSV linked by DeepChem's official Tox21 loader.
This section does three things: it verifies the provenance of the downloaded archive, it
prints the cleaning audit that turns raw records into modelling structures, and then it
**runs the preprocessing code live** on individual molecules so the standardization and
fingerprint steps can be read rather than taken on trust.

### 3.1 Provenance of the raw archive

The download metadata is committed in `data/raw/source.json`. The cell below prints it,
then **recomputes the SHA-256 of the committed archive** and re-reads the archive to count
its records, so the provenance claim is checked rather than quoted. You should see the
recorded and recomputed digests match, and the record count agree.

In [ ]:
# --- 3.1 Verify the provenance of the raw archive -------------------------------------
# NOTEBOOK_STARTED is taken here: after the project tree is in place (a fresh Colab clone
# writes every file, so an earlier timestamp would flag all of them), and before this
# notebook writes anything of its own. Section 12 uses it to prove write isolation.
NOTEBOOK_STARTED = time.time()

source = json.loads((PROJECT_ROOT / "data" / "raw" / "source.json").read_text())
raw_archive = PROJECT_ROOT / "data" / "raw" / "tox21.csv.gz"

print("Source URL          :", source["url"])
print("DeepChem loader     :", source["deepchem_loader_url"])
print("Downloaded (UTC)    :", source["downloaded_at_utc"])
print("Recorded SHA-256    :", source["sha256"])
recomputed = hashlib.sha256(raw_archive.read_bytes()).hexdigest()
print("Recomputed SHA-256  :", recomputed)
print("SHA-256 match       :", recomputed == source["sha256"])
print()
print("Recorded byte count :", "{:,}".format(source["bytes"]))
print("Byte count on disk  :", "{:,}".format(raw_archive.stat().st_size))
print("Recorded records    :", "{:,}".format(source["raw_records"]))

raw_frame = pd.read_csv(raw_archive)
print("Records re-read here:", "{:,}".format(len(raw_frame)))
print("Record count match  :", len(raw_frame) == source["raw_records"])
print()
print("Assay label columns in the file ({}):".format(len(source["assay_label_columns"])))
print(" ", ", ".join(source["assay_label_columns"]))
print("Assays selected for this benchmark:", ", ".join(ASSAYS))
print()
show(raw_frame.head(3), "First three raw records as downloaded")

### 3.2 Cleaning, duplicates, missing labels, and class imbalance

`data/audit/assay_counts.csv` is the non-overlapping accounting identity for each assay:
raw observed labels, rows dropped for invalid structures, structures removed for
conflicting duplicate labels, consistent duplicate rows merged, and the resulting clean
structure counts.

Three data characteristics matter for everything that follows, and the tables below show
all three:

* **Missing labels.** Tox21 is a sparse matrix — a compound is not tested in every assay. A
  missing assay label excludes a record only from *that* assay. Missing is never treated as
  inactive, which would manufacture false negatives.
* **Duplicate structures.** After standardization, the same structure can appear more than
  once. Where the duplicate rows agree on the label they are merged; where they conflict,
  the structure is removed from that assay entirely rather than arbitrating the disagreement.
* **Class imbalance**, the problem this project is about. Read `clean_prevalence` below:
  it is the fraction of tested compounds that are active, and it ranges from roughly 0.04 to
  0.16 across the three assays. A model predicting "inactive" for everything would score
  between 84.1% and 95.7% accuracy and find nothing, which is why accuracy appears nowhere in this
  notebook and average precision, recall at a fixed budget, Brier score and log loss do.

In [ ]:
# --- 3.2 The cleaning audit and the class imbalance -----------------------------------
assay_counts = pd.read_csv(PROJECT_ROOT / "data" / "audit" / "assay_counts.csv")

raw_view = assay_counts[["assay", "raw_records", "raw_observed", "raw_positives",
                         "raw_negatives", "raw_missing"]]
show(raw_view, "Raw label accounting in the downloaded file (note raw_missing: Tox21 is sparse)")

clean_view = assay_counts[["assay", "invalid_structure_observed_rows",
                           "valid_structure_observed_rows", "conflicting_structures",
                           "conflicting_rows", "consistent_duplicate_rows_merged",
                           "clean_structures", "clean_positives", "clean_negatives",
                           "clean_prevalence"]]
show(clean_view, "Cleaning outcome per assay (structure level)")

# State the imbalance explicitly, since it is the premise of the whole project.
imbalance = assay_counts[["assay", "clean_structures", "clean_positives", "clean_prevalence"]].copy()
imbalance["active_percent"] = (imbalance.clean_prevalence * 100).map("{:.2f}%".format)
imbalance["inactive_per_active"] = (
    (imbalance.clean_structures - imbalance.clean_positives) / imbalance.clean_positives
).map("{:.1f} : 1".format)
imbalance["accuracy_of_all_inactive"] = (1 - imbalance.clean_prevalence).map("{:.2%}".format)
show(imbalance[["assay", "clean_structures", "clean_positives", "active_percent",
                "inactive_per_active", "accuracy_of_all_inactive"]],
     "Class imbalance per assay, and why accuracy is not used as a metric anywhere below")

structure_counts = json.loads(
    (PROJECT_ROOT / "data" / "audit" / "structure_counts.json").read_text())
print("Global structure accounting (data/audit/structure_counts.json):")
for key, value in structure_counts.items():
    print("  {:<40} {}".format(key, value))
print()

verification = json.loads((PROJECT_ROOT / "docs" / "data_verification.json").read_text())
checks = verification["checks"]
print("Independent data audit (docs/data_verification.json):")
print("  partitions checked   :", verification["partitions_checked"])
print("  fingerprints checked :", verification["fingerprints_checked"])
print("  checks passing       : {} of {}".format(sum(bool(v) for v in checks.values()), len(checks)))
failed = [name for name, passed in checks.items() if not passed]
print("  checks failing       :", failed if failed else "none")

### 3.3 The preprocessing code, executed here

The committed `data/processed/` files were produced by `toxbench.data.prepare`. That
function writes into `data/`, so this notebook never calls it. Instead the cell below
imports the same pure functions `prepare` uses and runs them **on individual molecules,
in memory**, so the feature engineering is visible rather than described:

1. `standardize_smiles` applies RDKit `Cleanup`, keeps the largest carbon-containing
   fragment (so salts and counter-ions are discarded), neutralises charges with `Uncharger`,
   re-sanitises, and returns canonical isomeric SMILES plus the non-stereochemical
   Bemis-Murcko scaffold. Structure identity keeps stereochemistry; the scaffold used for
   grouping deliberately does not.
2. The `structure_id` is the SHA-256 of that canonical SMILES, which is what makes
   duplicate detection exact and partition membership reproducible.
3. The feature vector is a 2,048-bit Morgan fingerprint of radius 2, generated with the
   same `rdFingerprintGenerator` settings the pipeline uses.

The last column of the table is the check that matters: the fingerprint recomputed here is
compared bit-for-bit against the committed row of `data/processed/fingerprints.npy` for the
same structure. All six should read `True`. Acyclic molecules have no ring system and so
no Murcko scaffold; they are collected under the sentinel group `__EMPTY__`, which becomes
important in section 4.3.

In [ ]:
# --- 3.3 Run the standardization and fingerprint code on individual molecules ---------
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

from toxbench.data import SETTINGS, standardize_smiles

print("Preprocessing settings recorded with the committed data (toxbench.data.SETTINGS):")
for key, value in SETTINGS.items():
    print("  {:<28} {}".format(key, value))
print()

structures = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "structures.csv")
# mmap_mode keeps the 15 MB fingerprint matrix off the heap; only the rows touched are read.
fingerprints = np.load(PROJECT_ROOT / "data" / "processed" / "fingerprints.npy", mmap_mode="r")
print("Committed fingerprint matrix: {:,} structures x {:,} bits, dtype {}".format(
    fingerprints.shape[0], fingerprints.shape[1], fingerprints.dtype))
print("Committed structure table   : {:,} rows".format(len(structures)))
print()

generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=SETTINGS["fingerprint_radius"],
    fpSize=SETTINGS["fingerprint_bits"],
    includeChirality=SETTINGS["fingerprint_chirality"])

# Six molecules taken from the top of the raw file, chosen by position and not by outcome.
demo_rows = []
for raw_smiles in raw_frame.smiles.head(6):
    canonical, scaffold = standardize_smiles(raw_smiles)
    structure_id = hashlib.sha256(canonical.encode()).hexdigest()
    recomputed_fp = generator.GetFingerprintAsNumPy(Chem.MolFromSmiles(canonical)).astype(np.uint8)

    match = structures[structures.structure_id == structure_id]
    if len(match):
        committed_fp = np.asarray(fingerprints[int(match.fp_index.iloc[0])])
        identical = bool(np.array_equal(recomputed_fp, committed_fp))
    else:
        identical = None  # Structure excluded from the modelling set by the cleaning rules.

    demo_rows.append({
        "raw SMILES": raw_smiles,
        "canonical SMILES": canonical,
        "changed by standardization": raw_smiles != canonical,
        "Murcko scaffold": scaffold,
        "structure_id (first 12)": structure_id[:12],
        "bits set of 2048": int(recomputed_fp.sum()),
        "fingerprint matches committed": identical,
    })

show(pd.DataFrame(demo_rows), "Standardization and fingerprint generation, run in this notebook")

recomputed_flags = [r["fingerprint matches committed"] for r in demo_rows]
print("Molecules standardized here          :", len(demo_rows))
print("Fingerprints bit-identical to committed: {} of {}".format(
    sum(1 for f in recomputed_flags if f is True), len(recomputed_flags)))
print("Acyclic molecules (scaffold __EMPTY__) :",
      sum(1 for r in demo_rows if r["Murcko scaffold"] == SETTINGS["empty_scaffold"]))
print()
print("Scaffold group sizes over the whole cleaned structure set:")
group_sizes = structures.scaffold.value_counts()
print("  distinct scaffold groups          : {:,}".format(len(group_sizes)))
print("  structures in the __EMPTY__ group : {:,} ({:.1%} of all structures)".format(
    int(group_sizes.get(SETTINGS["empty_scaffold"], 0)),
    group_sizes.get(SETTINGS["empty_scaffold"], 0) / len(structures)))
print("  structures in the benzene group   : {:,} ({:.1%})".format(
    int(group_sizes.get("c1ccccc1", 0)), group_sizes.get("c1ccccc1", 0) / len(structures)))
print("  singleton scaffold groups         : {:,}".format(int((group_sizes == 1).sum())))

## 4. Partition design

Each assay is split independently into approximately 60% training, 20% calibration and
20% test. The **scaffold** design allocates whole non-stereochemical Bemis-Murcko scaffold
groups using only scaffold membership and group size — never activity labels — so no
scaffold appears in more than one partition. The **random** design is a stratified
reference split with the same proportions. All competing methods share identical partitions,
which is what makes the comparisons between them paired.

The calibration partition exists so that the sigmoid calibrator in section 7 is fitted on
data that neither trained the model nor scored it. This is the design's main defence against
leakage: test labels are used in evaluation only.

`partitions/partition_summary.csv` records the realised sizes, fractions, positives and
deviations from the 60/20/20 target for every assay, split kind, seed and partition.

In [ ]:
# --- 4.1 Realised partition sizes and fractions ---------------------------------------
partition_summary = pd.read_csv(PROJECT_ROOT / "partitions" / "partition_summary.csv")
split_design = json.loads((PROJECT_ROOT / "partitions" / "split_design.json").read_text())

print("Prespecified seeds            :", split_design["seeds"])
print("Target fractions              :", split_design["settings"]["fractions"])
print("Scaffold allocation rule      :", split_design["settings"]["scaffold_assignment"])
print("Cross-validation rule         :", split_design["settings"]["cv"])
print("Infeasible-split policy       :", split_design["settings"]["infeasible"])
print()
print("partition_summary.csv rows    :", len(partition_summary))

PARTITION_ORDER = ["train", "calibration", "test"]
final_parts = partition_summary[partition_summary.partition.isin(PARTITION_ORDER)].copy()
final_parts["partition"] = pd.Categorical(final_parts.partition, PARTITION_ORDER, ordered=True)
final_parts["assay"] = pd.Categorical(final_parts.assay, ASSAYS, ordered=True)

sizes = (final_parts
         .groupby(["assay", "split", "partition"], observed=True)
         .agg(n=("n", "first"),
              realised_fraction=("fraction", "first"),
              target_fraction=("target_fraction", "first"),
              deviation_pp=("deviation_percentage_points", "first"),
              scaffold_groups_min=("scaffolds", "min"),
              scaffold_groups_max=("scaffolds", "max"))
         .reset_index())
show(sizes, "Realised train/calibration/test sizes and fractions (identical across the three seeds)")

In [ ]:
# --- 4.2 Positives and realised prevalence per partition ------------------------------
positives = (final_parts
             .pivot_table(index=["assay", "split", "partition"], columns="seed",
                          values="positive", aggfunc="first", observed=True, sort=False)
             .astype(int)
             .reset_index())
positives.columns = [c if isinstance(c, str) else "positives_seed_{}".format(c)
                     for c in positives.columns]
show(positives, "Positive (active) compounds per partition, by seed")

prevalence = (final_parts
              .pivot_table(index=["assay", "split", "partition"], columns="seed",
                           values="prevalence", aggfunc="first", observed=True, sort=False)
              .reset_index())
prevalence.columns = [c if isinstance(c, str) else "prevalence_seed_{}".format(c)
                      for c in prevalence.columns]
show(prevalence, "Realised active fraction per partition, by seed")

exploratory_parts = final_parts[final_parts.exploratory]
print("Final partitions flagged exploratory (fewer than 20 positives):",
      len(exploratory_parts))
if len(exploratory_parts):
    show(exploratory_parts[["assay", "split", "seed", "partition", "n", "positive"]])

### 4.3 The scaffold-split caveat

The scaffold allocator is label-independent: it processes groups by decreasing size times a
seeded priority, and assigns each group to whichever partition has the largest remaining
absolute capacity. Two groups are far larger than any other — the single `__EMPTY__` group
holding every acyclic structure, and the benzene group `c1ccccc1`.

Under this fixed size-based rule, **both land in training for every assay and every seed**,
and they also dominate two of the three training cross-validation folds. The consequence,
computed below rather than asserted, is that a scaffold test partition describes performance
on the *remaining* held-out scaffold distribution — not on a uniform random draw of all
scaffold groups. This is a deliberate consequence of a pre-registered rule that never splits
a group and never moves one on the basis of labels, but it narrows what "performance on new
scaffolds" means throughout this notebook.

In [ ]:
# --- 4.3 Where the two giant scaffold groups land, and what they do to the CV folds ---
giant_rows = []
cv_rows = []
for assay in ASSAYS:
    for seed in SEEDS:
        manifest = pd.read_csv(PROJECT_ROOT / "partitions" / assay / "scaffold" / "{}.csv".format(seed),
                               keep_default_na=False)
        grouped = (manifest.groupby("scaffold")
                   .agg(group_size=("structure_id", "size"),
                        partitions=("partition", lambda s: "/".join(sorted(set(s)))))
                   .sort_values("group_size", ascending=False))
        for scaffold, row in grouped.head(2).iterrows():
            giant_rows.append({"assay": assay, "seed": seed,
                               "scaffold": scaffold or "(empty string)",
                               "group_size": int(row.group_size),
                               "partition(s)": row.partitions})
        train_rows = manifest[manifest.partition == "train"]
        for scaffold in ("__EMPTY__", "c1ccccc1"):
            member = train_rows[train_rows.scaffold == scaffold]
            if len(member):
                folds = member.cv_fold.value_counts().sort_index()
                cv_rows.append({"assay": assay, "seed": seed, "scaffold": scaffold,
                                "train_members": len(member),
                                "cv_folds_occupied": ", ".join(
                                    "fold {}: {}".format(int(f), int(c)) for f, c in folds.items())})

giants = pd.DataFrame(giant_rows)
show(giants, "Two largest scaffold groups per assay and seed, and where they are allocated")
print("Giant-group placements landing in training only:",
      int((giants["partition(s)"] == "train").sum()), "of", len(giants))
print()
show(pd.DataFrame(cv_rows), "Cross-validation fold occupancy of the two giant groups inside training")

cv_folds = partition_summary[partition_summary.partition == "cv_validation"]
scaffold_cv = cv_folds[cv_folds.split == "scaffold"]
print("Scaffold CV validation folds:", len(scaffold_cv))
print("Largest absolute deviation from equal thirds: {:.2f} percentage points".format(
    scaffold_cv.deviation_percentage_points.abs().max()))
print("Scaffold CV folds flagged exploratory (fewer than 20 positives):",
      int(scaffold_cv.exploratory.sum()))
show(scaffold_cv[scaffold_cv.exploratory][["assay", "seed", "cv_fold", "n", "positive", "prevalence"]],
     "Scaffold CV validation folds flagged exploratory")

random_cv = cv_folds[cv_folds.split == "random"]
print("Random CV validation folds:", len(random_cv),
      "| largest absolute deviation from equal thirds: {:.3f} percentage points".format(
          random_cv.deviation_percentage_points.abs().max()))

### 4.4 The splitter, re-run here

The partition manifests under `partitions/` are committed files. The cell below re-runs
`toxbench.splits.create_partition` — the function that produced them — in memory for one
assay under both split designs, and compares the result against the committed manifest row
by row. Nothing is written.

Two things should come out `True` for both designs: every structure receives the same
partition, and every training structure receives the same cross-validation fold. That is
what makes the committed partitions reproducible from the seed alone, and it demonstrates
the train/validation/test strategy the project uses.

In [ ]:
# --- 4.4 Re-run the partition code and compare against the committed manifests --------
from toxbench.splits import assign_groups, create_partition

RECHECK_ASSAY = ASSAYS[0]
assay_table = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "{}.csv".format(RECHECK_ASSAY))
print("Re-deriving the partitions for {} ({:,} cleaned structures, {:,} active)".format(
    RECHECK_ASSAY, len(assay_table), int(assay_table.label.sum())))
print()

recheck_rows = []
for split_kind in SPLIT_KINDS:
    for seed in SEEDS:
        # create_partition is a pure function of the table, the kind and the seed.
        rederived = create_partition(assay_table, split_kind, seed)
        committed = pd.read_csv(
            PROJECT_ROOT / "partitions" / RECHECK_ASSAY / split_kind / "{}.csv".format(seed),
            keep_default_na=False)
        merged = committed.merge(rederived[["structure_id", "partition", "cv_fold"]],
                                 on="structure_id", suffixes=("_committed", "_rederived"))
        recheck_rows.append({
            "assay": RECHECK_ASSAY, "split_kind": split_kind, "seed": seed,
            "structures_compared": len(merged),
            "all_structures_matched": len(merged) == len(committed) == len(rederived),
            "partition_identical": bool((merged.partition_committed == merged.partition_rederived).all()),
            "cv_fold_identical": bool((merged.cv_fold_committed == merged.cv_fold_rederived).all()),
        })

recheck = pd.DataFrame(recheck_rows)
show(recheck, "Committed partitions re-derived in this notebook from the seed alone")
print("Partition assignments reproduced exactly: {} of {}".format(
    int(recheck.partition_identical.sum()), len(recheck)))
print("Cross-validation folds reproduced exactly: {} of {}".format(
    int(recheck.cv_fold_identical.sum()), len(recheck)))
print()

# The allocator uses group sizes only. Demonstrate that on a small synthetic example:
# the same group sizes with the labels reversed produce exactly the same allocation.
toy_groups = ["A"] * 10 + ["B"] * 6 + ["C"] * 3 + ["D"] * 1
toy_assignment = assign_groups(toy_groups, [0.6, 0.2, 0.2], seed=17)
toy = pd.DataFrame({"scaffold_group": toy_groups,
                    "partition": np.array(PARTITION_ORDER)[toy_assignment]})
show(toy.groupby(["scaffold_group", "partition"], observed=True).size()
        .rename("structures").reset_index(),
     "assign_groups on a toy set of 4 groups: whole groups move together, labels never consulted")
print("assign_groups takes only (groups, fractions, seed) - no label argument exists,")
print("so the allocation cannot depend on activity. Signature:",
      "assign_groups(groups, fractions, seed)")

## 5. Live re-derivation of one partition

Sections 6 to 10 read results that were computed once and committed. This section runs the
project's own command line interface **now, in this session**, so the reader sees real code
produce real numbers rather than only reading tables from files.

The demonstration uses `configs/smoke.json`: a single assay (NR-AR), a single split kind
(scaffold), a single seed (17), both classifiers and all four balancing choices — eight
fitted models out of the benchmark's 144. It is a demonstration slice, not a re-run of the
benchmark, which takes far longer than a notebook session should.

Section 5.2 prints the implementation that does the fitting — the real source of
`toxbench/modeling.py`, read with `inspect.getsource` — so the training code can be read
here and not merely invoked.

**The full 144-model configuration is available as a documented switch.** Cell 5.1 defines
`RUN_FULL_BENCHMARK`, which defaults to `False`; at that default section 5 does exactly
what is described above. Setting it to `True` points cells 5.1, 5.3 and 5.4 at
`configs/full.json` instead — the same file that produced every committed result in
sections 6 to 10 — so the whole benchmark is re-derived here rather than one partition of
it. The two cells that run the pipeline read the same variable, so they cannot disagree
about which configuration is active.

What differs between the two files is the partition grid, not the method. Both search the
same hyperparameter grids over the same three cross-validation folds with the same two
classifiers and four balancing choices; `full.json` widens 1 assay x 1 split kind x 1 seed
to 3 x 2 x 3, which is 144 fitted models instead of 8 and **1,872 estimator fits instead of
104** (1,728 inside cross-validation plus 144 final refits). It also raises the bootstrap
replicates from 200 to 1,000 and the worker processes from 1 to 3.

The cost is real and is worth reading before flipping the flag. The committed run of
`configs/full.json` took **545.8 seconds of wall clock for the fitting step alone** on a
10-core machine (recorded in `outputs/full/invocations.jsonl`; 1,597 seconds of summed
per-model time, spread across 3 worker processes), and the evaluation step — 36 groups at
1,000 bootstrap replicates each — runs on top of that. A standard Colab CPU runtime
provides 2 vCPUs, so the same work there takes substantially longer and may exceed the
session. Leave the flag at `False` unless you intend to wait.

**Write isolation.** The run is directed at the scratch directory `outputs/notebook_demo/`.
The first cell asserts that the resolved output path is not, and is not inside,
`outputs/full/`, `outputs/smoke/`, `data/` or `partitions/`, and then clears the scratch
directory so the reported wall time is a cold run rather than a checkpoint resume. The
`prepare` command is never invoked, because that is the only command that writes to `data/`
and `partitions/`.

In [ ]:
# --- 5.1 Pick the configuration, assert write isolation, clear the scratch directory --
#
# FULL-BENCHMARK SWITCH -----------------------------------------------------------------
# False (the default): section 5 runs configs/smoke.json - one assay (NR-AR), one split
#     kind (scaffold), one seed (17), both classifiers and all four balancing choices.
#     Eight fitted models and 104 estimator fits: a demonstration slice, sized to
#     finish inside a notebook session.
# True: section 5 runs configs/full.json instead - the complete 144-model benchmark that
#     produced every committed result in sections 6 to 10. That is 1,872 estimator fits.
#     The committed run took 545.8 seconds of wall clock for the fitting step alone on a
#     10-core machine, with the 36-group, 1,000-replicate evaluation on top of it; a
#     2-vCPU Colab runtime has far less parallelism, so it takes substantially longer and
#     may exceed the session. Flip this only if you intend to wait.
# Cells 5.1, 5.3, 5.4, 5.5, 5.6 and 5.7 all read the variables set just below, so they
# cannot disagree about which configuration is active. Either way nothing outside
# outputs/notebook_demo/ is written, and section 12 verifies that afterwards.
RUN_FULL_BENCHMARK = False

DEMO_CONFIG_NAME = "full.json" if RUN_FULL_BENCHMARK else "smoke.json"
DEMO_CONFIG_PATH = PROJECT_ROOT / "configs" / DEMO_CONFIG_NAME
# Read from the file the command line will read, rather than reusing the dict from 2.5.
DEMO_CONFIG = json.loads(DEMO_CONFIG_PATH.read_text())

DEMO_OUTPUT = (PROJECT_ROOT / "outputs" / "notebook_demo").resolve()
PROTECTED = [(PROJECT_ROOT / p).resolve() for p in
             ("outputs/full", "outputs/smoke", "data", "partitions")]

assert DEMO_OUTPUT.name == "notebook_demo", DEMO_OUTPUT
assert PROJECT_ROOT in DEMO_OUTPUT.parents, "demo output must live inside the project"
for protected in PROTECTED:
    assert DEMO_OUTPUT != protected, "demo output must not be a protected directory"
    assert protected not in DEMO_OUTPUT.parents, "demo output must not be inside a protected directory"
    assert DEMO_OUTPUT not in protected.parents, "demo output must not contain a protected directory"

print("Demo output directory :", DEMO_OUTPUT)
print("Protected directories :")
for protected in PROTECTED:
    print("   ", protected)
print()
if not VERSIONS_MATCH_PINS:
    print("WARNING: installed package versions differ from the pins (see cell 2.3).")
    print("         The live results below may differ numerically from the committed run.")
    print()
print("{} configuration (configs/{}):".format(
    "Full" if RUN_FULL_BENCHMARK else "Smoke", DEMO_CONFIG_NAME))
print(json.dumps(DEMO_CONFIG, indent=2))
expected_models = (len(DEMO_CONFIG["assays"]) * len(DEMO_CONFIG["split_kinds"])
                   * len(DEMO_CONFIG["seeds"]) * len(DEMO_CONFIG["classifiers"])
                   * len(DEMO_CONFIG["balancing"]))
print("\nModels this demonstration will fit:", expected_models)

if DEMO_OUTPUT.exists():
    print("Clearing", DEMO_OUTPUT, "so the timing below is a cold run.")
    shutil.rmtree(DEMO_OUTPUT)

### 5.2 The training code itself, read out of the project source

The models in 5.3 are fitted by the project package rather than by code typed into this
notebook, so the cell below prints that implementation: the genuine source of
`toxbench/modeling.py`, obtained with `inspect.getsource`, plus the one function in
`toxbench/runner.py` that drives it. The cell only imports and introspects — it redefines
nothing, patches nothing and re-executes no training logic — so the text it prints is
exactly the code cell 5.3 then runs.

How the pieces fit together:

* **`tune_fit`** selects on training rows only. `frame` holds exactly the rows the
  manifest marks `train`, each carrying the `cv_fold` it was assigned when the partition
  was built; the fingerprint matrix `X` passed alongside it covers every structure, and is
  indexed only through those rows' `fp_index`. For each of the three folds it holds that fold out, resamples **the
  other two folds only**, fits every point of the hyperparameter grid on them, and scores
  each fit by average precision on the held-out fold — which is never resampled and never
  reweighted. The grid point with the highest mean average precision over the three folds
  is selected, the first grid entry winning ties.
* **`resample`** is what each fold calls. `none` and `weights` pass the rows through
  unchanged; `undersample` discards majority rows at random; `smoteenn` oversamples the
  minority with SMOTE and then cleans the result with edited nearest neighbours. It is
  never handed a held-out fold: during the search it sees the two folds being trained on,
  and after selection it sees the whole training partition for the refit. Its disk cache
  key is built from those exact rows and their order, so a cached array cannot cross a
  fold boundary.
* **Class weights are not a resampling step.** For the `weights` method `build_model` sets
  `class_weight='balanced'` on the estimator, so the reweighting happens inside the fit on
  the training folds — the same place, and the same fold isolation, as the resampling.
* Once a configuration is selected, `tune_fit` resamples the **whole** training partition
  once more and refits the winner on it. That refitted model is what produces the test
  predictions.
* **`SigmoidCalibrator`** is fitted afterwards, in `runner.run_partition`, on the model's
  scores for the separate **calibration** partition — never on training rows and never on
  test rows. Its slope is constrained to be non-negative, which is the monotonicity
  property section 7.1 relies on, and `run_partition` stores a `rank_order_identical` audit
  flag for every model alongside the calibrator's fitted slope and intercept.

The test partition takes no part in hyperparameter selection, in resampling, or in
calibration. Its labels are used only when the predictions written for it are scored.

In [ ]:
# --- 5.2 The training implementation, printed from the project's own source -----------
# Read-only by construction: this cell imports the module and uses inspect.getsource to
# print the real code. It defines no estimator, patches nothing and fits nothing - the
# fitting happens in cell 5.3, through the command line interface, in another process.
import ast
import inspect

import toxbench.modeling as modeling

MODELING_FILE = Path(inspect.getfile(modeling)).resolve()
RUNNER_FILE = (MODELING_FILE.parent / "runner.py").resolve()

print("Imported module  :", modeling.__name__)
print("Source file      :", MODELING_FILE)
print("Inside the project tree:", PROJECT_ROOT in MODELING_FILE.parents)
print("SHA-256          :", hashlib.sha256(MODELING_FILE.read_bytes()).hexdigest())
print("Module docstring :", inspect.getdoc(modeling))
print()


def print_source(name, source, caption, first_line=None):
    """Print one object's source under a short header naming it."""
    body = source.rstrip("\n")
    where = "" if first_line is None else "   lines {}-{}".format(
        first_line, first_line + len(body.splitlines()) - 1)
    print("=" * 94)
    print(name + where)
    print("   " + caption)
    print("=" * 94)
    print(body)
    print()


TRAINING_OBJECTS = [
    (modeling.resample,
     "Resampling. A sampler only ever sees the rows the caller passed in."),
    (modeling.grid_for,
     "The hyperparameter grid, taken from the configuration file, not hard coded."),
    (modeling.build_model,
     "The two estimators, and the single place class_weight='balanced' is set."),
    (modeling.fit_checked,
     "The .fit call itself, with a guard that turns non-convergence into an error."),
    (modeling.tune_fit,
     "Cross-validated selection inside the training partition, then the final refit."),
    (modeling.model_score,
     "The score the calibrator is fitted on and that defines the ranking."),
    (modeling.SigmoidCalibrator,
     "Sigmoid (Platt) calibration, fitted on calibration-partition rows only."),
]

for training_object, caption in TRAINING_OBJECTS:
    source_lines, first_line = inspect.getsourcelines(training_object)
    kind = "class" if inspect.isclass(training_object) else "function"
    print_source("toxbench.modeling.{}  [{}]".format(training_object.__qualname__, kind),
                 inspect.getsource(training_object), caption, first_line)

# toxbench.runner sets thread-count environment variables, MPLCONFIGDIR and a global
# scikit-learn option at import time, and cell 5.3 deliberately runs it in a separate
# process. So read run_partition out of the file with `ast` rather than importing it:
# still the committed source, with no import side effects in this kernel.
runner_text = RUNNER_FILE.read_text()
for node in ast.parse(runner_text).body:
    if isinstance(node, ast.FunctionDef) and node.name == "run_partition":
        print_source("toxbench.runner.run_partition  [function, read with ast, not imported]",
                     ast.get_source_segment(runner_text, node),
                     "Where the three partitions stay apart: tune_fit on train, the "
                     "calibrator on calibration, predictions on test.",
                     node.lineno)
        break
else:
    raise RuntimeError("run_partition not found in " + str(RUNNER_FILE))

# Name everything the module defines, so it is visible that nothing was quietly omitted.
defined_here = [name for name, obj in vars(modeling).items()
                if (inspect.isfunction(obj) or inspect.isclass(obj))
                and getattr(obj, "__module__", None) == modeling.__name__]
shown_above = [obj.__qualname__ for obj, _ in TRAINING_OBJECTS]
print("Objects defined in toxbench.modeling :", defined_here)
print("Printed in full above                :", shown_above)
print("Not printed (hashing/counting helpers):",
      [name for name in defined_here if name not in shown_above])

In [ ]:
# --- 5.3 Fit the demonstration models through the project CLI -------------------------
# The source printed in 5.2 is the code that runs here. Each model is tuned by 3-fold
# cross-validated average precision inside the training partition, with resampling and
# class weights fitted inside each fold; the calibrator is then fitted on the separate
# calibration partition. With RUN_FULL_BENCHMARK = False this fits eight models: expect
# well under a minute on a fast multi-core machine and up to a few minutes on a 2-vCPU
# Colab runtime. With RUN_FULL_BENCHMARK = True it fits all 144 - see the note in 5.1.
run_command = [sys.executable, "-m", "toxbench.runner", "run",
               "--root", str(PROJECT_ROOT),
               "--config", str(DEMO_CONFIG_PATH),
               "--output", str(DEMO_OUTPUT)]
print("$", " ".join(run_command))
print()

run_started = time.perf_counter()
run_proc = subprocess.run(run_command, cwd=str(PROJECT_ROOT), text=True, capture_output=True)
run_seconds = time.perf_counter() - run_started

print(run_proc.stdout.strip())
if run_proc.stderr.strip():
    print("--- stderr (last 2000 characters) ---")
    print(run_proc.stderr.strip()[-2000:])
print()
print("Return code              :", run_proc.returncode)
print("Cell wall time (seconds) : {:.1f}".format(run_seconds))
assert run_proc.returncode == 0, "the runner exited non-zero"

# The CLI records per-model failures in failure.json and still exits zero, so check status.
execution_status = json.loads((DEMO_OUTPUT / "execution_status.json").read_text())
print("execution_status.json    :", execution_status)
assert execution_status["failure_files"] == 0, "at least one demonstration model failed"
assert execution_status["completed_models"] == expected_models, "unexpected number of models"
print("\nAll {} demonstration models completed with no failures.".format(
    execution_status["completed_models"]))

In [ ]:
# --- 5.4 Evaluate the demonstration partition(s) through the project CLI --------------
# Reads the same DEMO_CONFIG_PATH as cell 5.3, so the two cells cannot disagree about
# which configuration is active. At the default that is 2 evaluation groups at 200
# bootstrap replicates; under RUN_FULL_BENCHMARK = True it is 36 groups at 1,000.
evaluate_command = [sys.executable, "-m", "toxbench.runner", "evaluate",
                    "--config", str(DEMO_CONFIG_PATH),
                    "--output", str(DEMO_OUTPUT)]
print("$", " ".join(evaluate_command))
print()

evaluate_started = time.perf_counter()
evaluate_proc = subprocess.run(evaluate_command, cwd=str(PROJECT_ROOT), text=True,
                               capture_output=True)
evaluate_seconds = time.perf_counter() - evaluate_started

print(evaluate_proc.stdout.strip())
if evaluate_proc.stderr.strip():
    print("--- stderr (last 2000 characters) ---")
    print(evaluate_proc.stderr.strip()[-2000:])
print()
print("Return code              :", evaluate_proc.returncode)
print("Cell wall time (seconds) : {:.1f}".format(evaluate_seconds))
assert evaluate_proc.returncode == 0, "the evaluator exited non-zero"
print("evaluation_status.json   :",
      json.loads((DEMO_OUTPUT / "evaluation_status.json").read_text()))
print()
print("Total live demonstration wall time: {:.1f} seconds".format(run_seconds + evaluate_seconds))

In [ ]:
# --- 5.5 The held-out test metrics this notebook just produced ------------------------
demo_metrics = pd.read_csv(DEMO_OUTPUT / "metrics.csv")
demo_view = demo_metrics[demo_metrics.calibration != "constant"].copy()
demo_view["method"] = pd.Categorical(demo_view.method, list(METHODS), ordered=True)
demo_view["calibration"] = pd.Categorical(demo_view.calibration, list(CALIBRATIONS), ordered=True)
demo_view = demo_view.sort_values(["classifier", "method", "calibration"])

columns = ["assay", "split_kind", "seed", "classifier", "method", "calibration",
           "average_precision", "roc_auc", "precision_top10", "recall_top10",
           "brier", "log_loss", "n", "n_positive", "prevalence", "selection_count",
           "exploratory"]
# Describe whatever the active configuration actually covers, rather than naming a
# partition that RUN_FULL_BENCHMARK = True would make only one of eighteen.
demo_partitions = [(assay, kind, seed) for assay in DEMO_CONFIG["assays"]
                   for kind in DEMO_CONFIG["split_kinds"] for seed in DEMO_CONFIG["seeds"]]
demo_label = ("{}, {}, seed {}".format(*demo_partitions[0]) if len(demo_partitions) == 1
              else "{} partitions from configs/{}".format(len(demo_partitions), DEMO_CONFIG_NAME))
show(demo_view[columns], "Held-out test metrics computed by the cells above ({})".format(demo_label))

constant_rows = demo_metrics[demo_metrics.calibration == "constant"].drop_duplicates(
    subset=["assay", "split_kind", "seed", "classifier", "calibration"])
show(constant_rows[["classifier", "calibration", "train_prevalence", "average_precision",
                    "roc_auc", "brier", "log_loss", "precision_top10", "recall_top10"]],
     "Reference predictor that always returns the training prevalence")
print("This constant predictor is the yardstick used throughout section 7: any model whose")
print("Brier score is worse than this one is producing probabilities of no practical value,")
print("however well it ranks.")

### 5.6 Does the live slice reproduce the committed benchmark?

`configs/smoke.json` uses the same hyperparameter grids, the same three cross-validation
folds and the same fixed partition as `configs/full.json` for this assay, split kind and
seed; only the bootstrap replicate count differs, and that affects interval width, not the
fitted models. So the eight models just fitted should reproduce the committed predictions
for `NR-AR / scaffold / seed 17`. Under `RUN_FULL_BENCHMARK = True` the live run
contains that partition too, as one of eighteen, and the same comparison is made on
it.

The comparison below reports the **maximum absolute difference** in predicted probability
per model, together with whether the two CSV files are byte-identical. Byte identity is
expected on the machine that produced the committed outputs; on different hardware or BLAS
builds, small floating-point differences can appear. It is therefore reported rather than
asserted — a difference of order 1e-16 is a floating-point artefact, not a reproduction
failure.

In [ ]:
# --- 5.6 Compare the live predictions against the committed benchmark predictions -----
# The committed benchmark under outputs/full/ covers all eighteen partitions; the live run
# covers one of them at the default flag and all eighteen when RUN_FULL_BENCHMARK = True.
# The comparison is therefore pinned to the partition both always contain.
COMPARISON_SLICE = ("NR-AR", "scaffold", 17)
compare_assay, compare_kind, compare_seed = COMPARISON_SLICE
slice_in_config = (compare_assay in DEMO_CONFIG["assays"]
                   and compare_kind in DEMO_CONFIG["split_kinds"]
                   and compare_seed in DEMO_CONFIG["seeds"])

if not slice_in_config:
    # Unreachable with either committed configuration; a guard for a hand-edited one.
    comparison = pd.DataFrame()
    print("configs/{} does not contain {} / {} / seed {}, so the live run has no".format(
        DEMO_CONFIG_NAME, compare_assay, compare_kind, compare_seed))
    print("counterpart to the committed predictions and nothing is compared here.")
    print("Sections 6 to 10 read the committed benchmark directly and are unaffected.")
else:
    comparison_rows = []
    for classifier in DEMO_CONFIG["classifiers"]:
        for method in DEMO_CONFIG["balancing"]:
            relative = (Path("runs") / compare_assay / compare_kind / str(compare_seed)
                        / classifier / method / "predictions.csv")
            live_path = DEMO_OUTPUT / relative
            committed_path = PROJECT_ROOT / "outputs" / "full" / relative
            live = pd.read_csv(live_path, keep_default_na=False)
            committed = pd.read_csv(committed_path, keep_default_na=False)
            same_order = live.structure_id.tolist() == committed.structure_id.tolist()
            same_labels = live.label.tolist() == committed.label.tolist()
            max_difference = max(
                float(np.abs(live[column].to_numpy() - committed[column].to_numpy()).max())
                for column in ("raw", "sigmoid"))
            comparison_rows.append({
                "classifier": classifier, "method": method,
                "test_rows": len(live),
                "same_structure_order": same_order,
                "same_labels": same_labels,
                "max_abs_probability_difference": max_difference,
                "byte_identical_file": (hashlib.sha256(live_path.read_bytes()).hexdigest()
                                        == hashlib.sha256(committed_path.read_bytes()).hexdigest()),
            })

    comparison = pd.DataFrame(comparison_rows)
    show(comparison, "Live demonstration predictions compared with the committed benchmark predictions")
    print("Largest absolute probability difference across all eight models: {:.3e}".format(
        comparison.max_abs_probability_difference.max()))
    print("Models whose prediction file is byte-identical to the committed file: {} of {}".format(
        int(comparison.byte_identical_file.sum()), len(comparison)))

In [ ]:
# --- 5.7 A figure produced by the live run --------------------------------------------
demo_panel = DEMO_OUTPUT / "evaluation" / "NR-AR" / "scaffold" / "17" / "forest"
if not demo_panel.is_dir():
    # Unreachable with either committed configuration; a guard for a hand-edited one.
    # show_image is deliberately not called, so section 12's missing-figure check stays
    # a real check rather than being tripped by a configuration choice made in 5.1.
    print("configs/{} produced no live evaluation panel for NR-AR / scaffold / 17 / forest.".format(
        DEMO_CONFIG_NAME))
    print("Section 9 regenerates the committed panel for this group instead.")
else:
    print("Files written by the live evaluation for this group:")
    for path in sorted(demo_panel.iterdir()):
        print("  {:<24} {:>10,} bytes".format(path.name, path.stat().st_size))
    print()
    show_image(demo_panel / "precision_recall.png",
               "Precision-recall panel produced by the live run (NR-AR, scaffold, seed 17, forest)")

## 6. Detection results from the full benchmark

Everything in this section and the next four is read from the committed outputs of the
completed 144-model benchmark under `outputs/full/`. Nothing is recomputed from the fitted
models; the files are the record of the run.

* `metrics.csv` — one row per fitted model per calibration state (144 x 3 = 432 rows).
* `summary.csv` — seed means, standard deviations and counts per
  assay / split kind / classifier / balancing method / calibration state.
* `bootstrap_differences.csv` — paired percentile intervals for each balancing method
  minus the unbalanced baseline, within each individual test partition.
* `similarity_metrics.csv` — the same metrics recomputed inside fixed Tanimoto bins.

"Detection" here means ranking actives into the fixed top-10% screening budget: average
precision and ROC-AUC summarise the whole ranking, while precision and recall at 10%
measure what a screening programme would actually recover.

In [ ]:
# --- 6.1 Load the committed benchmark outputs and check the run accounting ------------
FULL = PROJECT_ROOT / "outputs" / "full"

full_metrics = pd.read_csv(FULL / "metrics.csv")
full_summary = pd.read_csv(FULL / "summary.csv")
bootstrap_differences = pd.read_csv(FULL / "bootstrap_differences.csv")
similarity_metrics = pd.read_csv(FULL / "similarity_metrics.csv")
alignment_audit = pd.read_csv(FULL / "alignment_similarity_audit.csv")

full_execution_status = json.loads((FULL / "execution_status.json").read_text())
full_evaluation_status = json.loads((FULL / "evaluation_status.json").read_text())

print("outputs/full/metrics.csv                    : {:,} rows".format(len(full_metrics)))
print("outputs/full/summary.csv                    : {:,} rows".format(len(full_summary)))
print("outputs/full/bootstrap_differences.csv      : {:,} rows".format(len(bootstrap_differences)))
print("outputs/full/similarity_metrics.csv         : {:,} rows".format(len(similarity_metrics)))
print("outputs/full/alignment_similarity_audit.csv : {:,} rows".format(len(alignment_audit)))
print()
print("execution_status.json  :", full_execution_status)
print("evaluation_status.json :", full_evaluation_status)
print()
print("Distinct fitted models in metrics.csv :", full_metrics.groupby(
    ["assay", "split_kind", "seed", "classifier", "method"]).ngroups)
print("Calibration states per model          :", sorted(full_metrics.calibration.unique()))
print("Model failures during the run         :", full_execution_status["failure_files"])
print("Models flagged exploratory (test partition has fewer than 20 positives):",
      int(full_metrics[full_metrics.calibration == "raw"].exploratory.sum()))

# Fixed orderings reused by every table in sections 6 to 10.
HEADLINE_METRICS = ["average_precision", "roc_auc", "precision_top10", "recall_top10",
                    "brier", "log_loss"]
RANKING_METRICS = ["average_precision", "roc_auc", "precision_top10", "recall_top10"]
PROBABILITY_METRICS = ["brier", "log_loss"]
MODEL_KEYS = ["assay", "split_kind", "seed", "classifier", "method"]


def order_frame(frame):
    """Apply the fixed assay / split / classifier / method / calibration ordering."""
    out = frame.copy()
    for column, order in (("assay", ASSAYS), ("split_kind", SPLIT_KINDS),
                          ("classifier", CLASSIFIERS), ("method", list(METHODS))):
        if column in out:
            # astype(str) first: re-categorising an existing Categorical whose category
            # list differs from `order` is deprecated in pandas, even when every value fits.
            out[column] = pd.Categorical(out[column].astype(str), order, ordered=True)
    if "calibration" in out:
        values = out.calibration.astype(str)
        present = [c for c in list(CALIBRATIONS) + ["constant"] if c in set(values)]
        out["calibration"] = pd.Categorical(values, present, ordered=True)
    return out

### 6.2 Seed means and standard deviations

`summary.csv` aggregates the three seeds for each assay, split kind, classifier, balancing
method and calibration state. Each cell below is `mean +/- standard deviation (n = seeds)`.
The standard deviation describes variation across the three fixed seeds; section 11 explains
why three seeds are not independent replication.

Positive direction differs by metric: higher is better for average precision, ROC-AUC and
the precision and recall at the fixed 10% screening budget, while **lower** is better for
the Brier score and log loss.

One thing to watch for while reading these tables, which section 7.1 then establishes
formally: within a given assay / split / classifier / method block, the `raw` and `sigmoid`
rows carry **identical** average precision, ROC-AUC, precision@10% and recall@10% values.
Only the Brier score and log loss differ between them.

In [ ]:
# --- 6.2 Seed mean +/- SD for every assay, split kind, classifier and method ----------
def format_mean_sd(frame, metrics_wanted=HEADLINE_METRICS):
    """Render each metric as 'mean +/- SD (n=seeds)' from the summary.csv columns."""
    out = frame[["assay", "split_kind", "classifier", "method", "calibration"]].copy()
    for metric in metrics_wanted:
        mean = frame["{}_mean".format(metric)]
        std = frame["{}_std".format(metric)]
        count = frame["{}_count".format(metric)]
        out[metric] = [
            "undefined" if pd.isna(m) else "{:.4f} +/- {:.4f} (n={})".format(m, 0.0 if pd.isna(s) else s, int(c))
            for m, s, c in zip(mean, std, count)]
    out["prevalence"] = ["{:.4f}".format(v) for v in frame["prevalence_mean"]]
    return out


ordered_summary = order_frame(full_summary).sort_values(
    ["assay", "split_kind", "classifier", "method", "calibration"])

model_rows = ordered_summary[ordered_summary.calibration.isin(list(CALIBRATIONS))]
for assay in ASSAYS:
    for split_kind in SPLIT_KINDS:
        block = model_rows[(model_rows.assay == assay) & (model_rows.split_kind == split_kind)]
        show(format_mean_sd(block).drop(columns=["assay", "split_kind"]),
             "{} | {} split | seed mean +/- SD over seeds {}".format(
                 assay, split_kind, ", ".join(str(s) for s in SEEDS)))

In [ ]:
# --- 6.3 The constant-prevalence reference predictor ----------------------------------
# This reference is the same for every balancing method, so it is reported once per
# assay / split kind / classifier rather than repeated four times.
constant_reference = (ordered_summary[ordered_summary.calibration == "constant"]
                      .drop_duplicates(subset=["assay", "split_kind", "classifier"]))
show(format_mean_sd(constant_reference).drop(columns=["method", "calibration"]),
     "Reference predictor that always returns the training prevalence (one row per group)")
print("Its average precision equals the test prevalence and its ROC-AUC is 0.5 by")
print("construction: it ranks nothing. Its Brier score, however, is the bar that a useful")
print("set of predicted probabilities has to clear - see section 7.2.")

### 6.4 Balancing minus unbalanced, with paired percentile intervals

`bootstrap_differences.csv` holds, for each individual test partition, the difference
between a balancing method and the unbalanced baseline on exactly the same bootstrap
resamples. Scaffold tests resample whole scaffold groups with replacement; random tests
resample examples. Every comparison within a partition shares identical sampled counts,
which is what makes the difference paired rather than a comparison of two independent
estimates.

These intervals are **within-partition and conditional**: they describe uncertainty of the
already fitted models on the drawn test sample. They do not describe uncertainty from
retraining, they are not independent across the three seeds, and they are not adjusted for
the number of comparisons. Each seed therefore gets its own interval below; interval bounds
are never averaged across seeds.

For the reason given in section 7.1, the `raw` and `sigmoid` rows of a ranking metric carry
the same difference wherever calibration preserved the ordering.

The separate table that follows reports the mean and standard deviation of the *difference*
across the three seeds, which is a different quantity from the intervals.

In [ ]:
# --- 6.4 Per-seed paired differences with 95% percentile intervals --------------------
bd = order_frame(bootstrap_differences)
# The baseline itself never appears as a left-hand side here, so drop it from the ordering.
bd["method"] = pd.Categorical(bd.method.astype(str),
                              [m for m in METHODS if m != "none"], ordered=True)
bd["interval"] = ["{:+.4f} [{:+.4f}, {:+.4f}]".format(d, lo, hi)
                  for d, lo, hi in zip(bd.difference, bd.ci_lower, bd.ci_upper)]

print("Bootstrap unit by split kind:")
print(bd.groupby(["split_kind", "bootstrap_unit"], observed=True).size().to_string())
print()
print("Replicates per comparison :", sorted(int(v) for v in bd.n_bootstrap.unique()))
print("Comparisons               : {:,} rows = {} partitions x {} classifiers x {} methods x {} calibrations x {} metrics".format(
    len(bd), len(ASSAYS) * len(SPLIT_KINDS) * len(SEEDS), len(CLASSIFIERS),
    bd.method.nunique(), len(CALIBRATIONS), bd.metric.nunique()))
print("Rows flagged exploratory  :", int(bd.exploratory.sum()))
print()

for metric in HEADLINE_METRICS:
    block = bd[bd.metric == metric].sort_values(
        ["assay", "split_kind", "classifier", "method", "calibration"])
    table = block.pivot(index=["assay", "split_kind", "classifier", "method", "calibration"],
                        columns="seed", values="interval")
    table.columns = ["seed {}".format(c) for c in table.columns]
    show(table.reset_index(),
         "{}: method minus unbalanced, difference with 95% percentile interval, per seed".format(metric))

In [ ]:
# --- 6.5 Spread of the difference across the three seeds ------------------------------
seed_spread = (bd.groupby(["assay", "split_kind", "classifier", "method", "calibration", "metric"],
                          observed=True)["difference"]
               .agg(["mean", "std", "count"])
               .reset_index()
               .rename(columns={"mean": "difference_seed_mean",
                                "std": "difference_seed_sd",
                                "count": "seeds"}))
seed_spread = seed_spread[seed_spread.metric.isin(HEADLINE_METRICS)]
seed_spread["metric"] = pd.Categorical(seed_spread.metric, HEADLINE_METRICS, ordered=True)
seed_spread = seed_spread.sort_values(
    ["assay", "split_kind", "classifier", "method", "calibration", "metric"])
seed_spread["difference_seed_mean"] = seed_spread.difference_seed_mean.map("{:+.4f}".format)
seed_spread["difference_seed_sd"] = seed_spread.difference_seed_sd.map("{:.4f}".format)
show(seed_spread, "Mean and standard deviation of the difference across the three seeds "
                  "(separate from, and not a substitute for, the per-seed intervals above)")

undefined = bd[bd.invalid_count > 0]
print("Comparisons with at least one undefined bootstrap replicate:", len(undefined))
if len(undefined):
    show(undefined[["assay", "split_kind", "seed", "classifier", "method", "calibration",
                    "metric", "valid_count", "invalid_count", "n_bootstrap"]],
         "Undefined replicates are excluded per metric and counted here")
print("Complete long-form table: outputs/full/bootstrap_differences.csv ({:,} rows).".format(
    len(bootstrap_differences)))

### 6.6 What the deltas add up to

The per-seed intervals above are the evidence; this is the tally over them. For each
balancing method and split kind there are 18 comparisons — 3 assays x 2 classifiers x
3 seeds — and the table counts, within each, how often the difference points the favourable
way and how often an interval excludes zero in each direction.

The tally is computed on the **`raw` calibration state only**. Including both states would
double every count without adding evidence, because section 7.1 shows the two states carry
identical ranking metrics.

An interval excluding zero here is a descriptive observation about one test partition. It
is not a multiplicity-controlled significance claim — see limitation 5 in section 11.

In [ ]:
# --- 6.6 Tally the deltas across all 18 comparisons per method per split kind ---------
# "Favourable" means the direction that helps: higher for ranking metrics, lower for losses.
FAVOURABLE_IS_HIGHER = {m: True for m in RANKING_METRICS}
FAVOURABLE_IS_HIGHER.update({m: False for m in PROBABILITY_METRICS})

tally_rows = []
raw_deltas = bd[bd.calibration == "raw"]
for metric in HEADLINE_METRICS:
    higher_is_better = FAVOURABLE_IS_HIGHER[metric]
    for method in [m for m in METHODS if m != "none"]:
        for split_kind in SPLIT_KINDS:
            block = raw_deltas[(raw_deltas.metric == metric)
                               & (raw_deltas.method == method)
                               & (raw_deltas.split_kind == split_kind)]
            if higher_is_better:
                favourable_sign = int((block.difference > 0).sum())
                excludes_favourable = int((block.ci_lower > 0).sum())
                excludes_unfavourable = int((block.ci_upper < 0).sum())
            else:
                favourable_sign = int((block.difference < 0).sum())
                excludes_favourable = int((block.ci_upper < 0).sum())
                excludes_unfavourable = int((block.ci_lower > 0).sum())
            tally_rows.append({
                "metric": metric, "method": method, "split_kind": split_kind,
                "comparisons": len(block),
                "favourable_in_sign": favourable_sign,
                "intervals_excluding_0_favourable": excludes_favourable,
                "intervals_excluding_0_unfavourable": excludes_unfavourable,
                "mean_difference": "{:+.4f}".format(float(block.difference.mean())),
            })

tally = order_frame(pd.DataFrame(tally_rows))
tally["metric"] = pd.Categorical(tally.metric, HEADLINE_METRICS, ordered=True)
tally = tally.sort_values(["metric", "method", "split_kind"])
show(tally, "Tally over the 18 comparisons per method per split kind (raw state, "
            "3 assays x 2 classifiers x 3 seeds)")
print("Higher is favourable for the first four metrics; lower is favourable for brier and")
print("log_loss, and the counts above already account for that.")
print()

# The one combination with interval-supported detection gains, isolated.
focus = raw_deltas[(raw_deltas.metric == "average_precision")
                   & (raw_deltas.method == "weights")
                   & (raw_deltas.split_kind == "scaffold")
                   & (raw_deltas.classifier == "forest")]
print("Class weighting + random forest + scaffold split, average precision:")
print("  comparisons                        :", len(focus))
print("  favourable in sign                 :", int((focus.difference > 0).sum()))
print("  intervals excluding 0, favourable  :", int((focus.ci_lower > 0).sum()))
print("  intervals excluding 0, unfavourable:", int((focus.ci_upper < 0).sum()))
print("  mean difference                    : {:+.4f}".format(float(focus.difference.mean())))
show(focus[["assay", "seed", "difference", "ci_lower", "ci_upper"]]
     .assign(interval=lambda f: [ "{:+.4f} [{:+.4f}, {:+.4f}]".format(d, lo, hi)
                                  for d, lo, hi in zip(f.difference, f.ci_lower, f.ci_upper)])
     [["assay", "seed", "interval"]],
     "The nine comparisons behind that line")
print("Those gains appear on two of the three assays and are absent on the third, on three")
print("overlapping seeds against unadjusted intervals. Section 8.1 reads this further; the")
print("project report treats it as a direction worth following up, not an established effect.")

## 7. Calibration results

Balancing changes what a predicted probability means. A model trained on an artificially
balanced sample predicts as though actives were far more common than they are, so its
outputs are systematically too high. This section measures that distortion and what the
sigmoid calibrator — fitted on the separate calibration partition, never on test data —
does about it.

### 7.1 Calibration cannot change the ranking here, and it did not

The sigmoid fitted in this project is constrained to a **non-negative slope**. A monotone
non-decreasing map cannot reorder compounds, so it cannot change any metric that depends
only on the ranking. Every fitted model stores its slope, intercept and a
`rank_order_identical` audit flag in `training.json`; the cell below reads all 144 of them
and then checks the committed metrics directly.

This governs how every table in this notebook must be read. Where the ordering is preserved,
average precision, ROC-AUC, and precision and recall at the fixed 10% budget are the *same
number* before and after calibration. **An unchanged ranking metric is not a calibration
effect, and is never presented as one here.** Calibration's effects belong entirely to the
Brier score, the log loss and the reliability curves.

Two of the 144 models record `rank_order_identical: false`. That flag is a strict
permutation-equality test, and `docs/report.md` records the audit finding that in both cases
the sigmoid maps two or three compounds whose raw probabilities differ in the last unit in
the last place of a float64 onto a single value, and that every collapsed group is entirely
negatives. The maximum change in average precision and ROC-AUC, printed below, is the
decisive number: it is exactly 0.

In [ ]:
# --- 7.1 The fitted calibrators, and whether any ranking metric moved -----------------
calibration_rows = []
for path in sorted((FULL / "runs").glob("*/*/*/*/*/training.json")):
    assay, split_kind, seed, classifier, method = path.parts[-6:-1]
    record = json.loads(path.read_text())
    fitted = record["calibration"]
    calibration_rows.append({
        "assay": assay, "split_kind": split_kind, "seed": int(seed),
        "classifier": classifier, "method": method,
        "slope": fitted["slope"], "intercept": fitted["intercept"],
        "calibration_n": fitted["calibration_n"],
        "calibration_positive": fitted["calibration_positive"],
        "constant_mapping": fitted["constant_mapping"],
        "rank_order_identical": record["rank_order_identical"],
    })
calibrations = pd.DataFrame(calibration_rows)

print("Fitted sigmoid calibrators read from outputs/full/runs/.../training.json:", len(calibrations))
print("Fitted slope range    : {:.3f} to {:.3f}".format(calibrations.slope.min(), calibrations.slope.max()))
print("Negative slopes (a rank reversal, which would have to be disclosed):",
      int((calibrations.slope < 0).sum()))
print("Degenerate constant mappings:", int(calibrations.constant_mapping.sum()))
print("Models whose full tie-broken test ordering is unchanged by calibration: {} of {}".format(
    int(calibrations.rank_order_identical.sum()), len(calibrations)))
print()

exceptions = calibrations[~calibrations.rank_order_identical]
if len(exceptions):
    show(exceptions[["assay", "split_kind", "seed", "classifier", "method", "slope", "intercept"]],
         "The models whose strict tie-broken ordering differed after calibration")

# The decisive check: did any metric value actually move, over all committed models?
raw_rows = full_metrics[full_metrics.calibration == "raw"].set_index(MODEL_KEYS)
sigmoid_rows = full_metrics[full_metrics.calibration == "sigmoid"].set_index(MODEL_KEYS)
constant_rows_full = full_metrics[full_metrics.calibration == "constant"].set_index(MODEL_KEYS)

change_summary = []
for metric in RANKING_METRICS + PROBABILITY_METRICS:
    difference = (sigmoid_rows[metric] - raw_rows[metric]).abs()
    change_summary.append({
        "metric": metric,
        "kind": "ranking" if metric in RANKING_METRICS else "probability",
        "models_with_identical_value": int((raw_rows[metric] == sigmoid_rows[metric]).sum()),
        "models": len(difference),
        "max_abs_change": "{:.3e}".format(float(difference.max())),
    })
show(pd.DataFrame(change_summary),
     "Change from raw to sigmoid predictions, over all {} committed models".format(len(raw_rows)))
print("Maximum absolute raw-to-sigmoid change in average precision: {:.1f}".format(
    float((sigmoid_rows.average_precision - raw_rows.average_precision).abs().max())))
print("Maximum absolute raw-to-sigmoid change in ROC-AUC          : {:.1f}".format(
    float((sigmoid_rows.roc_auc - raw_rows.roc_auc).abs().max())))
print("Exactly zero, for all {} models. Calibration is therefore never credited with a".format(len(raw_rows)))
print("ranking improvement anywhere in this notebook or in the project report.")

### 7.2 Brier score and log loss, before and after calibration

Both are losses, so **lower is better** throughout this subsection.

The yardstick is the constant predictor from section 6.3, which assigns every compound the
training prevalence. It ranks nothing at all, so any model that scores *worse* than it on
Brier score is producing probabilities with no practical value, however well it ranks. The
first table counts how many of the 36 models per balancing method fall below that bar,
before and after calibration.

In [ ]:
# --- 7.2 Probability quality against the constant-prevalence reference ----------------
baseline_brier = constant_rows_full.brier
quality_rows = []
for state in CALIBRATIONS:
    state_rows = full_metrics[full_metrics.calibration == state].set_index(MODEL_KEYS)
    worse = state_rows.brier > baseline_brier.reindex(state_rows.index)
    per_method = worse.groupby(level="method").sum()
    for method in METHODS:
        quality_rows.append({"calibration": state, "method": method,
                             "models": int((state_rows.index.get_level_values("method") == method).sum()),
                             "brier_worse_than_constant": int(per_method.get(method, 0))})
quality = order_frame(pd.DataFrame(quality_rows)).sort_values(["calibration", "method"])
show(quality, "Models whose Brier score is worse than the constant-prevalence predictor")

# reindex on every comparison: baseline_brier is indexed over all 144 models, while some
# of the left-hand sides below cover only a subset of them.
raw_worse = int((raw_rows.brier > baseline_brier.reindex(raw_rows.index)).sum())
sigmoid_worse = int((sigmoid_rows.brier > baseline_brier.reindex(sigmoid_rows.index)).sum())
total_models = len(raw_rows)
print("Raw state     : {} of {} models are worse than the constant predictor.".format(raw_worse, total_models))
print("Sigmoid state : {} of {} models are worse than the constant predictor.".format(sigmoid_worse, total_models))

unbalanced_raw = raw_rows[raw_rows.index.get_level_values("method") == "none"]
print("Unbalanced models among the {} raw failures: {} of {}".format(
    raw_worse,
    int((unbalanced_raw.brier > baseline_brier.reindex(unbalanced_raw.index)).sum()),
    len(unbalanced_raw)))
print("Every raw model that falls below the constant predictor is a balanced model.")
print()

# Seed means for both losses in all three states, so the repair can be read directly.
loss_table = ordered_summary.sort_values(["assay", "split_kind", "classifier", "method", "calibration"])
show(format_mean_sd(loss_table[loss_table.calibration.isin(list(CALIBRATIONS))],
                    metrics_wanted=PROBABILITY_METRICS),
     "Brier score and log loss, seed mean +/- SD, before and after calibration (lower is better)")

# How large is the worst gap between a balanced method and the unbalanced baseline?
brier_deltas = bootstrap_differences[bootstrap_differences.metric == "brier"]
print("Largest Brier penalty of any balanced method against the unbalanced baseline:")
for state in CALIBRATIONS:
    block = brier_deltas[brier_deltas.calibration == state]
    print("  {:<8}: largest signed delta {:+.4f} | largest absolute delta {:.4f}".format(
        state, float(block.difference.max()), float(block.difference.abs().max())))
print()
print("Calibration closes that gap by an order of magnitude: the balanced models' raw")
print("probabilities are badly miscalibrated, and a two-parameter sigmoid fitted on a")
print("separate partition repairs most of the damage.")

### 7.3 What the reliability curves show

The committed reliability data
(`outputs/full/evaluation/<assay>/<split_kind>/<seed>/<classifier>/reliability.csv`, plotted
as `reliability.png` in section 9) place predictions into ten fixed bins of width 0.1 and
record, per bin, the count, the number of positives, the mean predicted probability and the
observed active fraction.

Two statistics are derived from those files below, both pooled over the three seeds:

* the **weighted gap** — the count-weighted mean of `|mean predicted - observed fraction|`
  over the non-empty bins, so a perfectly calibrated model scores 0;
* the **signed bias** — the same average without the absolute value, so a **positive** value
  means the model predicts higher probabilities than the observed rate, i.e. it is
  over-confident toward the positive class.

The signed bias column in the raw state is the mechanism behind section 7.2: balancing makes
models believe actives are common.

In [ ]:
# --- 7.3 Reliability gap and signed bias, pooled over seeds ---------------------------
reliability_rows = []
for path in sorted((FULL / "evaluation").glob("*/*/*/*/reliability.csv")):
    assay, split_kind, seed, classifier = path.parts[-5:-1]
    table = pd.read_csv(path)
    for (method, calibration), group in table.groupby(["method", "calibration"]):
        occupied = group[group["count"] > 0]          # empty bins carry no information
        weights = occupied["count"]
        signed = occupied.mean_prediction - occupied.observed_fraction
        reliability_rows.append({
            "assay": assay, "split_kind": split_kind, "seed": int(seed),
            "classifier": classifier, "method": method, "calibration": calibration,
            "abs_numerator": float((weights * signed.abs()).sum()),
            "signed_numerator": float((weights * signed).sum()),
            "denominator": float(weights.sum()),
            "occupied_bins": len(occupied),
        })

reliability = pd.DataFrame(reliability_rows)
pooled = (reliability.groupby(["assay", "split_kind", "classifier", "method", "calibration"])
          [["abs_numerator", "signed_numerator", "denominator"]].sum().reset_index())
pooled["weighted_gap"] = pooled.abs_numerator / pooled.denominator
pooled["signed_bias"] = pooled.signed_numerator / pooled.denominator

gap_table = (order_frame(pooled)
             .pivot_table(index=["assay", "split_kind", "classifier", "method"],
                          columns="calibration", values=["weighted_gap", "signed_bias"],
                          observed=True))
gap_table.columns = ["{} {}".format(a, b) for a, b in gap_table.columns]
show(gap_table.reset_index().sort_values(["assay", "split_kind", "classifier", "method"]),
     "Reliability gap and signed bias per group, pooled over the three seeds")

by_method = (order_frame(pooled).groupby(["method", "calibration"], observed=True)
             [["weighted_gap", "signed_bias"]].agg(["mean", "min", "max"]))
by_method.columns = ["{} {}".format(a, b) for a, b in by_method.columns]
show(by_method.reset_index(),
     "The same statistics averaged by balancing method ({} group-level values, {} per method)"
     .format(len(pooled), len(pooled) // (len(METHODS) * len(CALIBRATIONS))))

raw_pool = pooled[pooled.calibration == "raw"]
sig_pool = pooled[pooled.calibration == "sigmoid"]
merged_pool = raw_pool.merge(sig_pool, on=["assay", "split_kind", "classifier", "method"],
                             suffixes=("_raw", "_sigmoid"))
print("Groups whose weighted reliability gap improves after calibration: {} of {}".format(
    int((merged_pool.weighted_gap_sigmoid < merged_pool.weighted_gap_raw).sum()), len(merged_pool)))
print("Signed bias of balanced models, raw state   : {:+.4f} to {:+.4f}".format(
    float(raw_pool[raw_pool.method != "none"].signed_bias.min()),
    float(raw_pool[raw_pool.method != "none"].signed_bias.max())))
print("Signed bias of balanced models, sigmoid     : {:+.4f} to {:+.4f}".format(
    float(sig_pool[sig_pool.method != "none"].signed_bias.min()),
    float(sig_pool[sig_pool.method != "none"].signed_bias.max())))
print("Every balanced group is over-confident toward the positive class before calibration.")

### 7.4 Cases where calibration made things worse

Reporting only the successes would misrepresent the method, so the cell below finds every
model that came out worse on at least one of the two losses after calibration, with no
filtering.

In [ ]:
# --- 7.4 Every model that calibration made worse on at least one loss -----------------
degraded = []
for key in raw_rows.index:
    raw_row, sigmoid_row = raw_rows.loc[key], sigmoid_rows.loc[key]
    worse_brier = bool(sigmoid_row.brier > raw_row.brier)
    worse_log_loss = bool(sigmoid_row.log_loss > raw_row.log_loss)
    if worse_brier or worse_log_loss:
        assay, split_kind, seed, classifier, method = key
        degraded.append({
            "assay": assay, "split_kind": split_kind, "seed": seed,
            "classifier": classifier, "method": method,
            "brier_raw": round(float(raw_row.brier), 5),
            "brier_sigmoid": round(float(sigmoid_row.brier), 5),
            "log_loss_raw": round(float(raw_row.log_loss), 5),
            "log_loss_sigmoid": round(float(sigmoid_row.log_loss), 5),
            "made_worse": ("both" if worse_brier and worse_log_loss
                           else "brier only" if worse_brier else "log loss only"),
        })

degraded = order_frame(pd.DataFrame(degraded)).sort_values(
    ["assay", "split_kind", "seed", "classifier", "method"])
show(degraded, "Models where the sigmoid state is worse than the raw state on a loss")

print("Models made worse on at least one loss : {} of {}".format(len(degraded), total_models))
print("  worse on Brier score                 :", int(degraded.made_worse.isin(["brier only", "both"]).sum()))
print("  worse on log loss                    :", int(degraded.made_worse.isin(["log loss only", "both"]).sum()))
print("  worse on both                        :", int((degraded.made_worse == "both").sum()))
print()
print("Balancing methods represented among them:", sorted(set(degraded.method.astype(str))))
print("All of them are unbalanced models - the ones already close to well calibrated in the")
print("raw state, which therefore had nothing to gain and a little to lose from fitting a")
print("two-parameter map on a finite calibration partition. No balanced model was made worse.")

## 8. Scaffold versus random: does any benefit survive the harder split?

The research question is about **new** chemical scaffolds, so the comparison between the two
split designs is the point of the whole design, not a side analysis. The random split is the
reference: a stratified split with the same 60/20/20 proportions, where test compounds are
chemically similar to training compounds. The scaffold split holds out whole scaffold groups.

The cell below pairs the two within each assay, classifier and balancing method, using the
committed seed means. Three columns deserve attention:

* **AP / prevalence.** The scaffold test partitions have a somewhat different active
  fraction from the random ones, and average precision scales with prevalence. Dividing by
  the test prevalence removes that confound, so this ratio is the fairer comparison.
* **AP SD.** The across-seed standard deviation. Stability matters as much as the mean here:
  a method that is better on average but swings wildly across seeds is not usable.
* **Δ AP.** Negative means the scaffold split is harder.

Section 4.3 bounds the interpretation: the scaffold test set is the *remaining* scaffold
distribution after the two giant groups are placed in training, so "new scaffolds" has a
narrower meaning than the phrase suggests.

In [ ]:
# --- 8.1 Scaffold minus random, paired within assay, classifier and balancing method --
raw_summary = full_summary[full_summary.calibration == "raw"]
paired = raw_summary.pivot_table(
    index=["assay", "classifier", "method"], columns="split_kind",
    values=["average_precision_mean", "average_precision_std", "roc_auc_mean",
            "recall_top10_mean", "precision_top10_mean", "prevalence_mean"])

ap = paired["average_precision_mean"]
ap_sd = paired["average_precision_std"]
prev = paired["prevalence_mean"]
ratio = paired["average_precision_mean"] / prev

comparison_table = pd.DataFrame({
    "AP random": ap["random"].round(3),
    "AP scaffold": ap["scaffold"].round(3),
    "delta AP": (ap["scaffold"] - ap["random"]).round(3),
    "AP SD random": ap_sd["random"].round(3),
    "AP SD scaffold": ap_sd["scaffold"].round(3),
    "AP/prevalence random": ratio["random"].round(2),
    "AP/prevalence scaffold": ratio["scaffold"].round(2),
    "delta ROC-AUC": (paired["roc_auc_mean"]["scaffold"] - paired["roc_auc_mean"]["random"]).round(3),
    "delta recall@10%": (paired["recall_top10_mean"]["scaffold"] - paired["recall_top10_mean"]["random"]).round(3),
}).reset_index()
comparison_table = order_frame(comparison_table).sort_values(["assay", "classifier", "method"])
show(comparison_table, "Scaffold minus random, paired within assay, classifier and balancing method")

cells = len(ap)
print("Assay x classifier x method cells compared:", cells)
print()
print("Average precision lower on the scaffold split      : {} of {}  (mean change {:+.4f})".format(
    int((ap["scaffold"] < ap["random"]).sum()), cells, float((ap["scaffold"] - ap["random"]).mean())))
print("Prevalence-normalised ratio lower on scaffold      : {} of {}".format(
    int((ratio["scaffold"] < ratio["random"]).sum()), cells))
print("ROC-AUC lower on the scaffold split                : {} of {}  (mean change {:+.4f})".format(
    int((paired["roc_auc_mean"]["scaffold"] < paired["roc_auc_mean"]["random"]).sum()), cells,
    float((paired["roc_auc_mean"]["scaffold"] - paired["roc_auc_mean"]["random"]).mean())))
print("Recall at the 10% budget lower on scaffold         : {} of {}  (mean change {:+.4f})".format(
    int((paired["recall_top10_mean"]["scaffold"] < paired["recall_top10_mean"]["random"]).sum()), cells,
    float((paired["recall_top10_mean"]["scaffold"] - paired["recall_top10_mean"]["random"]).mean())))
print("Across-seed SD of AP larger on the scaffold split  : {} of {}".format(
    int((ap_sd["scaffold"] > ap_sd["random"]).sum()), cells))
print("  mean AP SD, random split   : {:.4f}".format(float(ap_sd["random"].mean())))
print("  mean AP SD, scaffold split : {:.4f}  (a factor of {:.2f})".format(
    float(ap_sd["scaffold"].mean()), float(ap_sd["scaffold"].mean() / ap_sd["random"].mean())))
print()
print("The prevalence-normalised ratio falls in every cell too, so the drop is not an")
print("artefact of the two split designs having different test prevalences.")
print()

# Does any balancing benefit survive the harder split? Compare mean deltas side by side.
survive = (raw_deltas[raw_deltas.metric == "average_precision"]
           .groupby(["method", "classifier", "split_kind"], observed=True)
           .agg(comparisons=("difference", "size"),
                mean_delta=("difference", "mean"),
                intervals_favourable=("ci_lower", lambda s: int((s > 0).sum())),
                intervals_unfavourable=("ci_upper", lambda s: int((s < 0).sum())))
           .reset_index())
survive["mean_delta"] = survive.mean_delta.map("{:+.4f}".format)
show(order_frame(survive).sort_values(["method", "classifier", "split_kind"]),
     "Average-precision delta against the unbalanced baseline, by method, classifier and split kind")
print("Read against section 6.6: the resampling methods are less negative on the scaffold")
print("split than on the random split, but 'less harmful on the harder split' is not a")
print("benefit surviving. The one combination that accumulates favourable intervals without")
print("unfavourable ones is class weighting with the random forest on the scaffold split.")

## 9. Figures

Two figure types are produced for each of the 36 evaluation groups (3 assays x 2 split
kinds x 3 seeds x 2 classifiers):

* a **precision-recall panel** showing all four balancing methods before and after sigmoid
  calibration, with the test prevalence drawn as a dashed reference line — the line a
  random ranker would achieve, so the vertical distance above it is what the model adds;
* a **reliability panel** showing observed active fraction against mean predicted
  probability in ten fixed equal-width bins, with the bin counts plotted underneath on a
  symmetric-log axis. A perfectly calibrated model lies on the diagonal; a curve sagging
  below it is over-confident.

Section 9.1 regenerates both panels **inside this notebook** using the project's own
`toxbench.evaluation.create_figures` on the committed `predictions_wide.csv`, writing to
`outputs/notebook_demo/figures/` and never back into `outputs/full/`. Section 9.2 then
cross-checks the regenerated reliability table against the committed one.

In [ ]:
# --- 9.1 Regenerate a precision-recall and a reliability panel in this notebook -------
import matplotlib

FIGURE_SOURCE = FULL / "evaluation" / "NR-AR" / "scaffold" / "17" / "forest"
FIGURE_DEST = DEMO_OUTPUT / "figures" / "NR-AR" / "scaffold" / "17" / "forest"
assert DEMO_OUTPUT in FIGURE_DEST.parents, "regenerated figures must stay in the scratch directory"

wide = pd.read_csv(FIGURE_SOURCE / "predictions_wide.csv", keep_default_na=False)
print("Source predictions:", FIGURE_SOURCE / "predictions_wide.csv")
print("Test compounds: {:,} | active: {:,} | prevalence: {:.4f}".format(
    len(wide), int(wide.label.sum()), float(wide.label.mean())))
print("Prediction columns:", [c for c in wide.columns if c.endswith(("_raw", "_sigmoid"))])
print()

# create_figures forces the Agg backend to write files. Restoring whatever the notebook
# was using afterwards is a courtesy to any later plotting, not a requirement, so neither
# reading nor restoring the backend is allowed to break this cell.
try:
    previous_backend = matplotlib.get_backend()
except Exception as exc:
    previous_backend = None
    print("Could not read the current matplotlib backend:", exc)

create_figures(wide, FIGURE_DEST, "NR-AR | scaffold | seed 17 | forest (regenerated in this notebook)")

if previous_backend is not None:
    try:
        matplotlib.use(previous_backend, force=True)
    except Exception as exc:
        print("Could not restore the previous matplotlib backend:", exc)

print("Regenerated into", FIGURE_DEST)
for path in sorted(FIGURE_DEST.iterdir()):
    print("  {:<24} {:>10,} bytes".format(path.name, path.stat().st_size))
print()
show_image(FIGURE_DEST / "precision_recall.png", "Precision-recall panel regenerated here")
show_image(FIGURE_DEST / "reliability.png", "Reliability panel regenerated here")

In [ ]:
# --- 9.2 Cross-check the regenerated reliability table against the committed one ------
regenerated_bins = pd.read_csv(FIGURE_DEST / "reliability.csv")
committed_bins = pd.read_csv(FIGURE_SOURCE / "reliability.csv")
key = ["method", "calibration", "bin"]
merged = regenerated_bins.merge(committed_bins, on=key, suffixes=("_new", "_committed"))
print("Reliability bin rows compared:", len(merged))
for column in ("count", "positives"):
    print("  identical {:<10}: {}".format(
        column, bool((merged[column + "_new"] == merged[column + "_committed"]).all())))
for column in ("mean_prediction", "observed_fraction"):
    difference = (merged[column + "_new"] - merged[column + "_committed"]).abs()
    print("  max abs difference in {:<18}: {:.3e}".format(column, float(difference.max(skipna=True))))
print()

empty_bins = committed_bins[committed_bins["count"] == 0]
print("Committed reliability bins for this group that contain no test compounds:", len(empty_bins))
print("Committed reliability bins flagged exploratory (fewer than 20 positives): {} of {}".format(
    int(committed_bins.exploratory.sum()), len(committed_bins)))
show(committed_bins[committed_bins.method == "none"][
    ["method", "calibration", "bin", "lower", "upper", "count", "positives",
     "mean_prediction", "observed_fraction", "exploratory"]],
     "Committed reliability bins for the unbalanced baseline of this group")

### 9.3 The committed panels

All 36 precision-recall panels and all 36 reliability panels are committed to the
repository, in PNG and PDF, under
`outputs/full/evaluation/<assay>/<split kind>/<seed>/<classifier>/`. Together they come to
roughly 24 MB, which makes a saved notebook containing all 72 slow to load and awkward to
share.

By default the cell below displays six of them rather than all 72, which keeps a saved
notebook small enough to open quickly. Three are the precision-recall panels named in `docs/report.md` (its section 3.4 --
the project report, not this notebook); three are reliability panels, and rather than
name those from memory the cell **selects them from the data** using the weighted gap
computed in section 7.3:

| Panel | How it is chosen |
| --- | --- |
| NR-AhR / scaffold / 42 / forest | precision-recall: named by the report as the clearest class-weighting gain |
| SR-MMP / random / 17 / forest | precision-recall: named by the report as a group where SMOTEENN visibly costs precision |
| NR-AR / scaffold / 2026 / logistic | precision-recall: named by the report as the high-variance group; the NR-AR scaffold logistic block carries the largest across-seed standard deviations in the benchmark (section 6.2) |
| selected in code | reliability: the random-split panel with the largest raw weighted gap |
| selected in code | reliability: the scaffold-split panel with the largest raw weighted gap |
| NR-AhR / scaffold / 17 / forest | reliability: the worked example used in `docs/report.md` section 4.4 |

The cell prints the gap behind each selected reliability panel, so the choice is visible
rather than asserted.

**To display all 72 panels instead**, set `SHOW_ALL_PANELS = True` at the top of the cell
and re-run it. Section 9.4 lists every committed figure file either way and confirms that
all of them are present, whether or not they are rendered inline.

In [ ]:
# --- 9.3 Display the committed panels -------------------------------------------------
# Set this to True to render all 36 precision-recall and all 36 reliability panels
# inline. That is roughly 24 MB of images and makes the saved notebook slow to open.
SHOW_ALL_PANELS = False

EVALUATION_GROUPS = [(assay, split_kind, seed, classifier)
                     for assay in ASSAYS
                     for split_kind in SPLIT_KINDS
                     for seed in SEEDS
                     for classifier in CLASSIFIERS]

# The three precision-recall panels named in docs/report.md section 3.4.
REPORT_PANELS = [
    ("NR-AhR", "scaffold", 42, "forest", "precision_recall"),
    ("SR-MMP", "random", 17, "forest", "precision_recall"),
    ("NR-AR", "scaffold", 2026, "logistic", "precision_recall"),
]

# The reliability panels are selected from the data, not named from memory: for each
# split design, the panel whose worst balancing method has the largest raw weighted gap.
per_panel = reliability[reliability.calibration == "raw"].copy()
per_panel["gap"] = per_panel.abs_numerator / per_panel.denominator
worst_method = per_panel.loc[
    per_panel.groupby(["assay", "split_kind", "seed", "classifier"])["gap"].idxmax()]

reliability_panels = []
for split_kind in SPLIT_KINDS:
    row = worst_method[worst_method.split_kind == split_kind].nlargest(1, "gap").iloc[0]
    reliability_panels.append((row.assay, row.split_kind, int(row.seed), row.classifier,
                               "largest raw weighted gap on the {} split: {:.4f} ({})".format(
                                   split_kind, row.gap, row.method)))
# Plus the worked example docs/report.md uses in its section 4.4.
worked = worst_method[(worst_method.assay == "NR-AhR") & (worst_method.split_kind == "scaffold")
                      & (worst_method.seed == 17) & (worst_method.classifier == "forest")].iloc[0]
reliability_panels.append((worked.assay, worked.split_kind, int(worked.seed), worked.classifier,
                           "worked example in docs/report.md section 4.4; worst raw gap here "
                           "{:.4f} ({})".format(worked.gap, worked.method)))

print("Reliability panels selected from the committed data:")
for assay, split_kind, seed, classifier, why in reliability_panels:
    print("  {} | {} | seed {} | {:<8} - {}".format(assay, split_kind, seed, classifier, why))
print()

REPORT_PANELS = REPORT_PANELS + [(a, k, s, c, "reliability")
                                 for a, k, s, c, _ in reliability_panels]

if SHOW_ALL_PANELS:
    selected = [(a, k, s, c, fig)
                for fig in ("precision_recall", "reliability")
                for (a, k, s, c) in EVALUATION_GROUPS]
    print("Displaying every committed panel: {} figures.".format(len(selected)))
else:
    selected = REPORT_PANELS
    print("Displaying {} panels: {} precision-recall panels named in the report, and {} "
          "reliability panels selected above from the committed data.".format(
              len(selected), len(selected) - len(reliability_panels), len(reliability_panels)))
    print("Set SHOW_ALL_PANELS = True at the top of this cell to render all {} committed "
          "panels instead.".format(2 * len(EVALUATION_GROUPS)))
print()

for assay, split_kind, seed, classifier, figure in selected:
    panel = FULL / "evaluation" / assay / split_kind / str(seed) / classifier
    label = "precision-recall" if figure == "precision_recall" else "reliability"
    show_image(panel / "{}.png".format(figure),
               "Committed {} panel: {} | {} split | seed {} | {}".format(
                   label, assay, split_kind, seed, classifier))

In [ ]:
# --- 9.4 Inventory of every committed figure file -------------------------------------
inventory = []
for group in sorted((FULL / "evaluation").glob("*/*/*/*")):
    if not group.is_dir():
        continue
    assay, split_kind, seed, classifier = group.parts[-4:]
    row = {"assay": assay, "split_kind": split_kind, "seed": int(seed), "classifier": classifier}
    for name in ("precision_recall.png", "precision_recall.pdf",
                 "reliability.png", "reliability.pdf"):
        row[name] = (group / name).is_file()
    row["displayed_above"] = any(
        (assay, split_kind, int(seed), classifier) == (a, k, s, c) for a, k, s, c, _ in selected)
    inventory.append(row)

inventory = order_frame(pd.DataFrame(inventory)).sort_values(
    ["assay", "split_kind", "seed", "classifier"])
show(inventory, "Every committed evaluation group and its figure files")

figure_columns = ["precision_recall.png", "precision_recall.pdf", "reliability.png", "reliability.pdf"]
print("Evaluation groups        :", len(inventory))
print("Figure files expected    :", len(inventory) * len(figure_columns))
print("Figure files found       :", int(inventory[figure_columns].sum().sum()))
print("Groups with a panel shown:", int(inventory.displayed_above.sum()))
assert int(inventory[figure_columns].sum().sum()) == len(inventory) * len(figure_columns), \
    "a committed figure file is missing from the repository"
print()
print("All committed figure files are present. The PDF version of every panel sits beside")
print("the PNG, under outputs/full/evaluation/<assay>/<split kind>/<seed>/<classifier>/.")

## 10. Nearest-neighbour similarity analysis

Holding scaffolds out does not guarantee that test fingerprints are *dissimilar* to the
training set — two different scaffolds can still carry very similar substituents. For every
test compound the pipeline records the maximum Tanimoto similarity of its Morgan fingerprint
to any training compound, and then recomputes all metrics inside four similarity bins:
[0, 0.4), [0.4, 0.6), [0.6, 0.8) and [0.8, 1.0]. The edges are hard-coded constants in
`toxbench/postprocess.py`, not tunable parameters; `docs/references.md` records the rule
that similarity thresholds are fixed before outcomes are viewed.

This analysis is what connects sections 8 and 6: if performance depends on similarity to the
training set, and the scaffold split shifts test mass toward the low-similarity end, then
part of "the scaffold split is harder" is explained by *where the test compounds sit*
rather than by anything about scaffolds as such.

Bins behave differently from the full test set: a bin containing **no active compounds
yields an undefined average precision and ROC-AUC by design** — the pipeline records them
as missing rather than substituting zero, since substituting zero would silently penalise a
model for a property of the bin — and any subset with fewer than 20 actives is flagged
exploratory.

In [ ]:
# --- 10.1 The fixed bins, and how unfamiliar the test compounds actually are ----------
print("Fixed similarity bins present in similarity_metrics.csv:")
bin_edges = (similarity_metrics[["similarity_low", "similarity_high"]]
             .drop_duplicates().sort_values("similarity_low"))
for low, high in bin_edges.itertuples(index=False):
    print("  [{:.1f}, {:.1f}{}".format(low, high, "]" if high >= 1.0 else ")"))
print()

similarity_metrics["bin"] = [
    "[{:.1f}, {:.1f}{}".format(low, high, "]" if high >= 1.0 else ")")
    for low, high in zip(similarity_metrics.similarity_low, similarity_metrics.similarity_high)]
BIN_ORDER = list(bin_edges.assign(label=[
    "[{:.1f}, {:.1f}{}".format(low, high, "]" if high >= 1.0 else ")")
    for low, high in zip(bin_edges.similarity_low, bin_edges.similarity_high)]).label)

audit_view = order_frame(alignment_audit).sort_values(
    ["assay", "split_kind", "seed", "classifier"])
show(audit_view, "Per-group test size, actives, and nearest-training-compound similarity "
                 "(outputs/full/alignment_similarity_audit.csv)")

# The headline contrast: how much less familiar are the scaffold test compounds?
by_split = alignment_audit.groupby("split_kind")[["similarity_mean", "similarity_median"]].mean()
print("Mean nearest-training-compound Tanimoto similarity, by split design:")
for split_kind in SPLIT_KINDS:
    print("  {:<10}: mean {:.4f} | median {:.4f}".format(
        split_kind, float(by_split.loc[split_kind, "similarity_mean"]),
        float(by_split.loc[split_kind, "similarity_median"])))
print("The scaffold split does not invent a different problem so much as it loads the")
print("test set with compounds that are less like anything the model has seen.")

In [ ]:
# --- 10.2 Bin occupancy: where the test compounds actually fall ----------------------
# Occupancy is a property of the test partition, identical for every balancing method and
# classifier, so it is read once from the unbalanced raw rows of the first classifier.
occupancy = similarity_metrics[(similarity_metrics.method == "none")
                               & (similarity_metrics.calibration == "raw")
                               & (similarity_metrics.classifier == CLASSIFIERS[0])].copy()
occupancy = order_frame(occupancy)
occupancy["bin"] = pd.Categorical(occupancy["bin"], BIN_ORDER, ordered=True)

counts_table = occupancy.pivot(index=["assay", "split_kind", "seed"], columns="bin", values="n")
positives_table = occupancy.pivot(index=["assay", "split_kind", "seed"], columns="bin",
                                  values="n_positive")
show(counts_table.reset_index(), "Test compounds per fixed similarity bin")
show(positives_table.reset_index(), "Active compounds per fixed similarity bin")

# Share of each test partition in the lowest-similarity bin, by split design.
share = counts_table.div(counts_table.sum(axis=1), axis=0).reset_index()
share_by_split = share.groupby("split_kind", observed=True)[BIN_ORDER[0]].mean()
print("Mean share of the test partition in the lowest-similarity bin {}:".format(BIN_ORDER[0]))
for split_kind in SPLIT_KINDS:
    if split_kind in share_by_split:
        print("  {:<10}: {:.1%}".format(split_kind, float(share_by_split[split_kind])))
print()

print("Similarity-bin result rows                    : {:,}".format(len(similarity_metrics)))
print("Rows whose bin contains no actives            : {:,}".format(
    int((similarity_metrics.n_positive == 0).sum())))
print("Rows with undefined average precision         : {:,}".format(
    int(similarity_metrics.average_precision.isna().sum())))
print("Rows with undefined ROC-AUC                   : {:,}".format(
    int(similarity_metrics.roc_auc.isna().sum())))
print("Undefined rows that are exactly the zero-active rows:",
      bool((similarity_metrics.average_precision.isna()
            == (similarity_metrics.n_positive == 0)).all()))
print("Rows flagged exploratory (fewer than 20 actives): {:,} of {:,}".format(
    int(similarity_metrics.exploratory.sum()), len(similarity_metrics)))

In [ ]:
# --- 10.3 Metrics by similarity bin, and the gradient across the bins ----------------
sim = order_frame(similarity_metrics)
sim["bin"] = pd.Categorical(sim["bin"], BIN_ORDER, ordered=True)

for metric in ("average_precision", "recall_top10"):
    aggregated = (sim.groupby(["assay", "split_kind", "classifier", "method", "calibration", "bin"],
                              observed=True)
                  .agg(seed_mean=(metric, "mean"),
                       seeds_defined=(metric, "count"),
                       actives_min=("n_positive", "min"),
                       actives_max=("n_positive", "max"))
                  .reset_index())
    aggregated["value"] = [
        "undefined" if defined == 0 else "{:.4f} (seeds={}, actives {}-{})".format(mean, defined, lo, hi)
        for mean, defined, lo, hi in zip(aggregated.seed_mean, aggregated.seeds_defined,
                                         aggregated.actives_min, aggregated.actives_max)]
    table = aggregated.pivot(index=["assay", "split_kind", "classifier", "method", "calibration"],
                             columns="bin", values="value")
    show(table.reset_index(),
         "{} by fixed similarity bin, averaged over the seeds where it is defined".format(metric))

# Is the gradient real? Count the series where the third bin beats the first.
series = (sim[sim.calibration == "raw"]
          .pivot_table(index=["assay", "split_kind", "classifier", "method"],
                       columns="bin", values=["average_precision", "roc_auc"],
                       aggfunc="mean", observed=True))
low_bin, mid_bin, high_bin = BIN_ORDER[0], BIN_ORDER[1], BIN_ORDER[2]
total_series = len(series)
print("Assay x split x classifier x method series examined:", total_series)
for metric in ("average_precision", "roc_auc"):
    block = series[metric]
    rises = int((block[high_bin] > block[low_bin]).sum())
    monotone = int(((block[mid_bin] > block[low_bin]) & (block[high_bin] > block[mid_bin])).sum())
    print("  {:<18}: {} of {} series score higher in {} than in {}; "
          "monotone across the first three bins in {}".format(
              metric, rises, total_series, high_bin, low_bin, monotone))
print()
print("The top bin {} is not included in that count: on scaffold splits it holds only a".format(BIN_ORDER[3]))
print("handful of compounds, every such bin is flagged exploratory, and two of them contain")
print("no actives at all - which is where all {} undefined values come from.".format(
    int(similarity_metrics.average_precision.isna().sum())))
print()
print("Complete per-seed table: outputs/full/similarity_metrics.csv ({:,} rows).".format(
    len(similarity_metrics)))

## 11. Limitations

These are the limitations documented for this benchmark in `docs/data_methods.md`,
`docs/references.md`, `docs/implementation_plan.md` and `docs/report.md`. The cell after
this list prints the counts that each item refers to, straight from the committed files.

**Design and replication**

1. **Three seeds are not independent replication.** The seeds reshuffle the same cleaned
   compounds, so the resulting test partitions overlap in chemistry and are not independent
   datasets. Seed standard deviations describe split and fit variation within one dataset;
   they do not support population-level inference, and they say nothing about generalisation
   to other assays.
2. **The scaffold allocation is fixed and size-driven.** The giant `__EMPTY__` and benzene
   groups land in training for every assay and seed (section 4.3), so a scaffold test
   partition describes performance on the remaining held-out scaffold distribution, not a
   uniform random draw of scaffold groups. The same giant groups dominate two of the three
   training cross-validation folds. No group was split or moved to improve these figures.
3. **Some scaffold cross-validation folds carry very few actives**, for every seed, and are
   flagged exploratory. Scaffold cross-validation fold fractions deviate from equal thirds
   by several percentage points, while the final 60/20/20 partitions are within a small
   fraction of a percentage point of target. The cell below prints both figures.

**Uncertainty**

4. **The bootstrap intervals are descriptive and conditional.** They condition on the
   already fitted models and on the current test partition. They do not quantify uncertainty
   from retraining, from alternative partitions or from the choice of assay.
5. **The intervals are not multiplicity-adjusted.** The benchmark makes a large number of
   paired comparisons across assays, split kinds, classifiers, methods, calibration states
   and metrics. An interval that excludes zero here is a descriptive observation about one
   partition, not a multiplicity-controlled significance claim.
6. **Intervals are per partition and must not be pooled.** Section 6.4 reports each seed's
   interval separately; averaging interval bounds across seeds would not be a valid interval.
7. **The two classifiers' intervals within a partition are not independent evidence.** The
   bootstrap RNG seed is derived in `toxbench/postprocess.py` from the assay, split kind and
   partition seed, and **excludes the classifier**, so within a partition both classifiers
   draw the identical resample index vectors. The pairing within each classifier is correct
   and the intervals are valid, but a result appearing for both classifiers in the same
   partition is not two independent confirmations.
8. **Undefined metric values are recorded, not imputed.** Average precision and ROC-AUC are
   undefined for a resample or a subset with a single class, and the pipeline reports valid
   and invalid replicate counts rather than substituting zero.

**Small positive counts**

9. **Any subset with fewer than 20 actives is exploratory**, and the committed files carry an
   explicit `exploratory` flag for exactly that reason. This applies to several similarity
   bins, to reliability bins, and to some cross-validation folds. NR-AR is the limiting case,
   with test partitions holding 60 to 96 actives.

**Data and measurement**

10. **Assay labels are noisy, and the three endpoints are experimental.** An active result in
    NR-AR, NR-AhR or SR-MMP is a laboratory measurement, **not a determination that a
    chemical is harmful to people.**
11. **Fingerprint similarity is a limited measure of chemical relatedness.** Morgan
    fingerprints ignore stereochemistry even though structure identity here preserves it,
    tautomers are not canonicalised so residual near-duplicates can remain, and scaffold
    separation does not imply that all test fingerprints have low similarity to training
    examples. Section 10 reports the similarity distributions for that reason.
12. **Synthetic SMOTEENN rows are numerical training vectors, not new molecules**, and a
    1:1 SMOTE target does not guarantee 1:1 counts after the nearest-neighbour cleaning step.
13. **The design space is narrow.** One fingerprint, one calibration family, small
    hyperparameter grids, two classifiers, three of the twelve Tox21 assays. Results may not
    transfer to other descriptors, to isotonic or beta calibration, or to the assays not
    studied.

**Scope of the literature claims**

14. **Reference [4] is not treated as confirmed.** Only the bibliographic metadata and the
    publisher abstract of the ToxBench article were verified; `docs/references.md` records
    that the proposal's specific assertion about its tree-model class weighting could not be
    verified from the accessible article text, and it is not asserted here.
15. **The cited balancing studies are motivation, not baselines.** Their curation,
    descriptors, tuning and evaluation differ from this benchmark, so their numbers are not
    directly comparable to these.

**Execution**

16. **Local CPU execution replaced the proposed Colab CPU execution** for the full
    benchmark; this notebook supports Colab reproduction of the committed results and of a
    live slice. The initially installed RDKit 2026.09.2 failed basic Murcko-scaffold
    extraction on the execution host, so RDKit was pinned to 2025.09.3 before any analysis.

In [ ]:
# --- 11.1 The counts behind the limitation list, read from the committed files --------
print("Seeds used                                   :", SEEDS)
print("Fitted models                                :", full_execution_status["completed_models"])
print("Model failures                               :", full_execution_status["failure_files"])
print("Evaluation groups                            :", full_evaluation_status["partition_classifier_groups"])
print("Bootstrap replicates per comparison          :", full_evaluation_status["bootstrap_replicates"])
print()
print("Giant scaffold groups allocated to training  : {} of {} placements".format(
    int((giants["partition(s)"] == "train").sum()), len(giants)))
print("Scaffold CV validation folds flagged exploratory:", int(scaffold_cv.exploratory.sum()))
print("Smallest positive count in a scaffold CV validation fold:", int(scaffold_cv.positive.min()))
print("Largest scaffold CV deviation from equal thirds: {:.2f} percentage points".format(
    scaffold_cv.deviation_percentage_points.abs().max()))
print("Largest final-partition deviation from 60/20/20: {:.3f} percentage points".format(
    final_parts.deviation_percentage_points.abs().max()))
print()
print("Paired comparisons in bootstrap_differences.csv: {:,}".format(len(bootstrap_differences)))
print("  of which flagged exploratory                 : {:,}".format(int(bootstrap_differences.exploratory.sum())))
print("  with at least one undefined replicate        : {:,}".format(
    int((bootstrap_differences.invalid_count > 0).sum())))
print()
print("Similarity-bin rows                          : {:,}".format(len(similarity_metrics)))
print("  flagged exploratory                        : {:,}".format(int(similarity_metrics.exploratory.sum())))
print("  with no actives, so undefined AP and AUC   : {:,}".format(
    int((similarity_metrics.n_positive == 0).sum())))
print()
print("Test partitions with fewer than 20 actives   :", int(
    full_metrics[full_metrics.calibration == "raw"]
    .drop_duplicates(subset=["assay", "split_kind", "seed"]).exploratory.sum()))
print("Smallest number of actives in any test partition:",
      int(full_metrics[full_metrics.calibration == "raw"].n_positive.min()))
print()
print("Reminder: assay activity is an experimental endpoint, not a determination of human toxicity.")

## 12. Headline results and integrity check

### 12.1 What the benchmark found

The cell below recomputes, from the committed files this notebook has already loaded, the
figures that the project report's conclusion rests on. They are printed together so the
reader can check the report's claims against the data in one place rather than assembling
them from sections 6 to 10.

In words, and with the hedges the evidence requires:

* **On detection.** Of the three balancing methods tested, only inverse-frequency class
  weighting avoided systematically reducing the number of actives found within the fixed 10%
  budget. Random undersampling and SMOTEENN reduced detection more often than they improved
  it under both split designs, most clearly on the random split.
* **On new scaffolds.** The one combination with interval-supported gains was class
  weighting with the random forest on the scaffold split. Those gains come from two of the
  three assays and are absent on the third, on three overlapping seeds, against unadjusted
  intervals, and on a scaffold test set that excludes the two largest scaffold groups by
  construction. It is a direction worth testing further, **not an established effect**.
* **On probabilities.** Balancing degrades predicted probabilities badly and consistently;
  sigmoid calibration fitted on a separate partition repairs most of the damage. It is not
  costless — a small number of already well-calibrated unbalanced models came out slightly
  worse.
* **On what calibration does not do.** It did not improve ranking and could not have.
* **On the harder split.** Testing on held-out scaffolds was harder in every cell, and
  across-seed variability more than doubled.

No balancing method is described anywhere in this notebook as "best". The defensible reading
of this benchmark, stated in the report's conclusion, is to train without resampling, use
inverse-frequency class weights if anything, and calibrate on a separate partition before
treating any output as a probability.

In [ ]:
# --- 12.1 The headline numbers, recomputed from the committed files -------------------
headline = []


def record(statement, value):
    headline.append({"result": statement, "value": value})


ap_deltas = raw_deltas[raw_deltas.metric == "average_precision"]
record("Fitted models in the benchmark", "{} ({} failures)".format(
    full_execution_status["completed_models"], full_execution_status["failure_files"]))
record("Max |raw - sigmoid| average precision, all models",
       "{:.1f}".format(float((sigmoid_rows.average_precision - raw_rows.average_precision).abs().max())))
record("Max |raw - sigmoid| ROC-AUC, all models",
       "{:.1f}".format(float((sigmoid_rows.roc_auc - raw_rows.roc_auc).abs().max())))
record("Models with Brier worse than constant predictor, raw",
       "{} of {}".format(raw_worse, total_models))
record("Models with Brier worse than constant predictor, sigmoid",
       "{} of {}".format(sigmoid_worse, total_models))
record("Models made worse by calibration on a loss",
       "{} of {} (all unbalanced)".format(len(degraded), total_models))
for method in [m for m in METHODS if m != "none"]:
    for split_kind in SPLIT_KINDS:
        block = ap_deltas[(ap_deltas.method == method) & (ap_deltas.split_kind == split_kind)]
        record("Average precision delta, {} on the {} split".format(method, split_kind),
               "mean {:+.4f} | {} of {} intervals exclude 0 favourably, {} unfavourably".format(
                   float(block.difference.mean()), int((block.ci_lower > 0).sum()),
                   len(block), int((block.ci_upper < 0).sum())))
record("Class weighting + forest + scaffold split",
       "mean {:+.4f} | {} of {} intervals exclude 0 favourably, {} unfavourably".format(
           float(focus.difference.mean()), int((focus.ci_lower > 0).sum()), len(focus),
           int((focus.ci_upper < 0).sum())))
record("Average precision lower on the scaffold split",
       "{} of {} cells (mean {:+.4f})".format(
           int((ap["scaffold"] < ap["random"]).sum()), cells,
           float((ap["scaffold"] - ap["random"]).mean())))
record("Across-seed AP standard deviation, random -> scaffold",
       "{:.4f} -> {:.4f} (factor {:.2f})".format(
           float(ap_sd["random"].mean()), float(ap_sd["scaffold"].mean()),
           float(ap_sd["scaffold"].mean() / ap_sd["random"].mean())))
record("Series where AP rises from the lowest to the third similarity bin",
       "{} of {}".format(int((series["average_precision"][high_bin]
                              > series["average_precision"][low_bin]).sum()), total_series))
record("Undefined similarity-bin values (bins with no actives)",
       "{} average precision, {} ROC-AUC".format(
           int(similarity_metrics.average_precision.isna().sum()),
           int(similarity_metrics.roc_auc.isna().sum())))
record("Fitted sigmoid slope range (negative would be a rank reversal)",
       "{:.3f} to {:.3f}, none negative".format(calibrations.slope.min(), calibrations.slope.max()))

show(pd.DataFrame(headline), "Headline results, recomputed here from outputs/full/")

### 12.2 Files this notebook read and wrote

Read only, from the committed project tree: `requirements.txt`, `configs/full.json`,
`configs/smoke.json`, `data/raw/source.json`, `data/raw/tox21.csv.gz`,
`data/processed/structures.csv`, `data/processed/fingerprints.npy`,
`data/processed/<assay>.csv`, `data/audit/assay_counts.csv`,
`data/audit/structure_counts.json`, `docs/data_verification.json`,
`partitions/partition_summary.csv`, `partitions/split_design.json`,
`partitions/<assay>/<split kind>/<seed>.csv`, and everything under `outputs/full/`.

Written, all inside the scratch directory `outputs/notebook_demo/`: the demonstration
models and their evaluation from section 5 (eight at the default
`RUN_FULL_BENCHMARK = False`, all 144 if it is set to `True`), and the figure panels regenerated in section 9.

The final cell proves that. It lists what was written to the scratch directory, then walks
every protected directory and reports any file modified since the timestamp taken in cell
3.1 — before this notebook wrote anything. It also asserts that every figure requested by a
display cell was found.

In [ ]:
# --- 12.2 Prove that nothing outside the scratch directory was written ----------------
print("Scratch directory used by this notebook:", DEMO_OUTPUT)
written = sorted(p for p in DEMO_OUTPUT.rglob("*") if p.is_file())
print("Files written there: {:,}".format(len(written)))
print("Top-level entries:")
for path in sorted(DEMO_OUTPUT.iterdir()):
    print("  ", path.name + ("/" if path.is_dir() else ""))
print()

# Compare every protected file's modification time against the moment recorded in cell 3.1.
print("Files modified since this notebook started, per protected directory:")
untouched = True
for protected in PROTECTED:
    recent = [p for p in protected.rglob("*")
              if p.is_file() and p.stat().st_mtime > NOTEBOOK_STARTED]
    untouched = untouched and not recent
    print("  {:<58} {:>5} modified   (files scanned: {:,})".format(
        str(protected.relative_to(PROJECT_ROOT)), len(recent),
        sum(1 for p in protected.rglob("*") if p.is_file())))
    for path in recent[:10]:
        print("      ", path)
print()
print("All protected directories untouched by this notebook:", untouched)
assert untouched, "this notebook modified a protected directory"

print("Figures a display cell could not find:", len(MISSING_FIGURES))
for path in MISSING_FIGURES[:10]:
    print("   MISSING:", path)
assert not MISSING_FIGURES, "a figure this notebook tried to display is missing"
print()
print("Notebook started at (epoch seconds): {:.3f}".format(NOTEBOOK_STARTED))
print("Every result above was produced from the public repository at")
print("https://github.com/mzohaibcc/project606 and the committed files it contains.")